In [1]:
import numpy
import sys
import os
import matplotlib.pyplot as plt
# import skimage
import nrrd
from scipy import ndimage
import meshio
from skimage.morphology import skeletonize
from scipy.interpolate import splprep, splev
from sklearn.decomposition import PCA
from scipy.spatial import cKDTree
from scipy.interpolate import UnivariateSpline
import trimesh
from scipy.ndimage import median_filter

from scipy.spatial import cKDTree
from scipy.interpolate import UnivariateSpline
from skimage import measure

import pyvista as pv

from scipy.ndimage import uniform_filter1d

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from meshing_functions import getSurfaceMesh, export_centerline,  tetra_mesh_from_stl, set_the_offset, tetra_shell_from_two_surfaces


# pixel_spacing = numpy.loadtxt("/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/pixel_spacing.txt")
# base = "/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/"

pixel_spacing = numpy.loadtxt("/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/pixel_spacing.txt")
base = "/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/"


in_folder = "segment_w_lens/"    

folder = "Radial-profiles-from-mask/"

if not os.path.isdir(base + folder):
    os.mkdir(base + folder)


In [2]:
distance_A = -2.4
expansion_B = 2.4
distance_B = -10

# contact_axis_1 = 10*pixel_spacing[0]
# contact_axis_2 = 3.5*pixel_spacing[0]

contact_axis_1 = 8*pixel_spacing[0]
contact_axis_2 = 5*pixel_spacing[0]

h = 1.0


# distance_A = -2.2
# expansion_B = 2.4
# distance_B = -10

## Load segments, pixel spacing etc

In [3]:

# filename = "Segmentation_1.seg.nrrd"
filename = "segmentations_pdt1_v160725.seg.nrrd"

export_name = base + folder + filename[:-8]

# eyeball_id = 1
# muslce_ids = [2,3,4,5]
# pairs = [(0,2), (1,3)]

# lense_id = 7

eyeball_id = 3
muslce_ids = [4,5,6,7]
pairs = [(0,1), (2,3)]

lense_id = 2

background_id = -1


data, header = nrrd.read(base + in_folder + filename)
print(data.shape)



id_max = numpy.max(data)

print("pixel spacing = ",pixel_spacing)
print("id = ", id_max )

print("dimensions = ", data.shape)

if pixel_spacing[2]!= pixel_spacing[1]:
    print("resampling")
    factor = pixel_spacing[2]/pixel_spacing[1]
    data = ndimage.zoom(data, [1,1,factor], order = 0)


print("dimensions = ", data.shape)

id = int(numpy.max(data))
print("id = ", id)




(145, 209, 101)
pixel spacing =  [0.3125 0.3125 0.5   ]
id =  7
dimensions =  (145, 209, 101)
resampling
dimensions =  (145, 209, 162)
id =  7


## Create initial meshes

In [4]:
muscles = []

for id in range(1,numpy.max(data)+1):
    if id != background_id:
    # for id in range(3,4):
        print("id = ", id)


        if len(numpy.where((data>id-0.5) & (data<id+0.5 ))[0])>0:

            mask = numpy.zeros(data.shape, dtype=numpy.uint8)
            mask[(data>id-0.5) & (data<id+0.5 )]=1

            print(len(numpy.where(mask==1)[0]))

            numpy.save(export_name +"_filtered_surf_id_"+str(id), mask)

            if id ==eyeball_id:

                mask = ndimage.median_filter(mask, size=1)
                mask[mask<0.5]=0
                mask[mask>=0.5]=1

                mesh_surf = getSurfaceMesh(mask, export_name +"_surf_id_"+str(id)+".stl", pixel_spacing[0], True )

                mask_voxels = numpy.argwhere(mask == 1)  # shape (N, 3), coordinates in (z, y, x)
                # compute mean voxel position
                offset = mask_voxels.mean(axis=0)*pixel_spacing[0]  # [z_mean, y_mean, x_mean]
                numpy.savetxt(base + folder + "offset.txt", offset)

                mask_eye = mask

            if id in muslce_ids:
                print("Muscle")

                muscles.append(mask)
                mesh_surf = getSurfaceMesh(mask, export_name +"_surf_id_"+str(id)+".stl", pixel_spacing[0], False )

            # if id == lense_id:
            else:
                mask = ndimage.median_filter(mask, size=1)
                mask[mask<0.5]=0
                mask[mask>=0.5]=1

                mesh_surf = getSurfaceMesh(mask, export_name +"_surf_id_"+str(id)+".stl", pixel_spacing[0], False )
                


id =  1
25127
Problem edges: 2
Open Hole Edges: 0
Non-Manifold Junctions: 2
 -> Remaining Problem edges: 46
written file:  /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Radial-profiles-from-mask/segmentations_pdt1_v160725._surf_id_1.stl
id =  2
2778
Problem edges: 8
Open Hole Edges: 0
Non-Manifold Junctions: 8
 -> Remaining Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Radial-profiles-from-mask/segmentations_pdt1_v160725._surf_id_2.stl
id =  3
258428
Problem edges: 0
Outer and inner surfaces written.
Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Radial-profiles-from-mask/segmentations_pdt1_v160725._surf_id_3.stl
id =  4
23220
Muscle
Problem edges: 12
Open Hole Edges: 0
Non-Manifold Junctions: 12
 -> Remaining Problem edges: 0
written file:  /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Radial-profiles-from-mask/segmentations

## Lense axis

In [5]:
mesh = trimesh.load(export_name +"_surf_id_"+str(lense_id)+".stl")
# mesh.vertices = (mesh.vertices + offset) / pixel_spacing

lense_center = mesh.vertices.mean(axis=0)

print("lense_center = ", lense_center)

X = mesh.vertices - lense_center

_, _, vh = numpy.linalg.svd(X)

axis_direction = vh[-1]  # first principal axis


def axis(t):
    return lense_center + t * axis_direction


axis_points = [axis(-20.0), axis(20.0)]

export_centerline(axis_points, 1.0 , base + folder + "lense_axis.vtp", [0,0,0])

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # 

target_point =  axis(distance_A)/pixel_spacing[0]

print("target_point = ", target_point)

lense_center =  [21.07714597  8.39832615 26.44359753]
target_point =  [67.90096257 34.53607004 84.90012487]


## Section curve

- eyeball mesh ∩ place perpendicular to lense axis
- output: section curve

In [6]:
eyeball_mesh = trimesh.load(export_name +"_surf_id_"+str(eyeball_id)+"_outer.stl")

section = eyeball_mesh.section(
    plane_origin=target_point*pixel_spacing[0],
    plane_normal=axis_direction
)

section_curve = []

for entity in section.entities:
    idx = entity.points  # ordered indices
    
    for i in range(len(idx) - 1):
        section_curve.append([idx[i], idx[i+1]])

meshio.write(
    base + folder + "section_curve_A.vtk",
    meshio.Mesh(
        points=section.vertices,
        cells=[("line", numpy.array(section_curve))]
    )
)

## Section curve B & radial expansion


In [7]:
target_point_B =  axis(distance_B)/pixel_spacing[0]

section_B = eyeball_mesh.section(
    plane_origin=target_point_B*pixel_spacing[0],
    plane_normal=axis_direction
)

section_curve_B = []

for entity in section_B.entities:
    idx = entity.points  # ordered indices
    
    for i in range(len(idx) - 1):
        section_curve_B.append([idx[i], idx[i+1]])


# Center point in physical space 
center_physical = target_point_B * pixel_spacing[0]

original_vertices = numpy.array(section_B.vertices)

# Radial vectors from the center 
radial_vectors = original_vertices - center_physical

magnitudes = numpy.linalg.norm(radial_vectors, axis=1, keepdims=True)
normalized_vectors = radial_vectors / (magnitudes + 1e-8)



# Expand outward by xx mm 
expansion_distance = expansion_B 
expanded_vertices = original_vertices + (normalized_vectors * expansion_distance)

section_B.vertices = expanded_vertices

# 6. Rebuild the section_curve line segments using the new expanded coordinates
expanded_section_curve = []
for entity in section_B.entities:
    idx = entity.points  # ordered indices within this specific entity
    
    for i in range(len(idx) - 1):
        # We map the local entity indices directly to our expanded_vertices array
        expanded_section_curve.append([idx[i], idx[i+1]])

meshio.write(
    base + folder + "section_curve_B.vtk",
    meshio.Mesh(
        points=section_B.vertices,
        cells=[("line", numpy.array(expanded_section_curve))]
    )
)

## Functions: 
### Fit centerlines with/without additional constraints

In [8]:
def fit_target_constrained_spline(binned_centers, target_point, order, extension_factor = 0.05, smoothing_factor=3.5):
    """
    binned_centers: (N, 3) array of centers from the mask
    target_point: (3,) array [z, y, x] where the object should end
    """
    # 1. Append the target point to the end of your centers
    # We ensure target_point is shaped correctly for concatenation
    target_point = numpy.array(target_point).reshape(1, 3)
    constrained_points = numpy.concatenate([binned_centers, target_point], axis=0)
    
    # 2. Assign Weights (w)
    # We give the target point a very high weight (e.g., 10) 
    # and the noisy data points a lower weight (e.g., 1)
    weights = numpy.ones(len(constrained_points))
    weights[-1] = 20.0  # Force the spline to prioritize hitting the target
    
    print("constrained_points = ", constrained_points)

    # 3. Fit the Spline
    # We use the transpose .T because splprep expects (3, N)
    try:
        # Note: 'w' is the weights, 's' is the overall smoothing
        tck, u = splprep(constrained_points.T, w=weights, s=smoothing_factor, k=order)
        
        # 4. Evaluate from 0 to 1
        # Since the target is now the last point in the data, 
        # u=1.0 will correspond to the target point.
        u_new = numpy.linspace(-extension_factor, 1.0, 160)
        smooth_centerline = numpy.array(splev(u_new, tck)).T
        
        return smooth_centerline
    except Exception as e:
        print(f"Constrained spline fit failed: {e}")
        return None
    

def fit_3_constrained_spline(binned_centers, target_1, target_2, target_3, order, extension_factor = 0.05, smoothing_factor=3.5):
    """
    Fits a spline by truncating binned centers at target_1 to avoid over-fitting,
    then sequentially appends target_2 and target_3.
    
    Ordering: [Body Part 1 (Up to Target 1)] -> [Target 1] -> [Target 2] -> [Target 3]
    """
    # 1. Standardize shapes
    target_1 = numpy.array(target_1).reshape(1, 3) # Intersects mid-body (Truncation point)
    target_2 = numpy.array(target_2).reshape(1, 3) # Intersects near end
    target_3 = numpy.array(target_3).reshape(1, 3) # Eye axis anchor (Absolute End)

    # 2. Find where target_1 belongs along the binned centers
    distances = numpy.linalg.norm(binned_centers - target_1, axis=1)
    split_idx = numpy.argmin(distances)

    # 3. Keep ONLY the binned centers that come strictly before target_1
    # We discard binned_centers[split_idx:] to prevent the spline from fight-fitting noise
    binned_truncated = binned_centers[:split_idx]

    # 4. Assemble the final sequence sequentially
    constrained_points = numpy.concatenate([
        binned_truncated,
        target_1,
        target_2,
        target_3
    ], axis=0)

    # 5. Assign precise weights based on the new truncated layout
    weights = numpy.ones(len(constrained_points))
    
    target_1_final_idx = len(binned_truncated)
    target_2_final_idx = len(constrained_points) - 2
    target_3_final_idx = len(constrained_points) - 1

    weights[target_1_final_idx] = 10.0  # Lock onto Mid-plane target
    weights[target_2_final_idx] = 10.0  # Lock onto End-plane target
    weights[target_3_final_idx] = 20.0  # Lock onto Eye anchor target

    print(f"Truncated muscle body from {len(binned_centers)} down to {len(binned_truncated)} centers.")
    print("Clean Truncated constrained_points = \n", constrained_points)

    # 6. Fit the Spline
    try:
        tck, u = splprep(constrained_points.T, w=weights, s=smoothing_factor, k=order)
        
        u_new = numpy.linspace(-extension_factor, 1.0, 160)
        smooth_centerline = numpy.array(splev(u_new, tck)).T
        
        return smooth_centerline
    except Exception as e:
        print(f"Truncated 3-point constrained spline fit failed: {e}")
        return None


def fit_dual_constrained_spline(binned_centers, target_1, target_2, order, extension_factor = 0.05, smoothing_factor=3.5):
    """
    binned_centers: (N, 3) array of centers from the mask
    target_1: (3,) array where the object should begin
    target_2: (3,) array where the object should end
    """
    # 1. Prepare the sequence: [Start, Binned Data, End]
    points = [
        binned_centers,
        numpy.array(target_1).reshape(1, 3),
        numpy.array(target_2).reshape(1, 3)
    ]
    constrained_points = numpy.concatenate(points, axis=0)
    
    # 2. Assign Weights
    # Weights[0] is Start, Weights[-1] is End. 
    # High values force the spline to hit those exact coordinates.
    weights = numpy.ones(len(constrained_points))
    weights[0] = 50.0  # Force start
    weights[-1] = 50.0 # Force end
    
    # 3. Fit the Spline
    # We pass the weights array to 'w'
    try:
        tck, u = splprep(constrained_points.T, w=weights, s=smoothing_factor, k=order)
        
        # 4. Evaluate
        # u=0.0 will be target_start, u=1.0 will be target_end
        u_new = numpy.linspace(-extension_factor, 1.0, 160)
        smooth_centerline = numpy.array(splev(u_new, tck)).T
        
        return smooth_centerline
    except Exception as e:
        print(f"Dual constrained spline fit failed: {e}")
        return None

def fit_analytical_centerline(mask, target_point, ignored_nodes, order, n_bins=20, extension_factor=0.2):
        # 1. Get all voxel coordinates (Z, Y, X)
        points = numpy.argwhere(mask == 1).astype(float)
        
        if len(points) < 10:
            return None

        # 2. Find the main orientation using PCA
        pca = PCA(n_components=1)
        # Project points onto the first principal component 
        projection = pca.fit_transform(points)


        # =========================================================================
        # 2.5 FORCE ORIENTATION TOWARD EYE 
        # =========================================================================
        if len(target_point) > 0:
            # The direction of the PCA axis vector in 3D space
            pca_axis = pca.components_[0] 
            
            # Vector pointing from the muscle center of mass to the target point
            center_of_mass = points.mean(axis=0)
            direction_to_target = numpy.array(target_point[-1]) - center_of_mass
            
            # If the dot product is negative, the PCA axis points AWAY from the target.
            if numpy.dot(pca_axis, direction_to_target) < 0:
                print("reverse")
                projection = -projection

        
        # 3. Sort points based on their position along the principal axis
        sort_idx = numpy.argsort(projection.flatten())

        sort_idx = sort_idx[int(len(sort_idx)*ignored_nodes):int(len(sort_idx)*(1-ignored_nodes))]

        sorted_points = points[sort_idx]
        sorted_projection = projection[sort_idx].flatten()

        # 4. Create "Binned Centers" to handle the noise
        # Fit to the average center of 'n' slices
        bins = numpy.linspace(sorted_projection.min(), sorted_projection.max(), n_bins)
        binned_centers = []
        
        for i in range(len(bins)-1):
            # Find points that fall within this slice of the object
            mask_slice = (sorted_projection >= bins[i]) & (sorted_projection < bins[i+1])
            if numpy.any(mask_slice):
                center = sorted_points[mask_slice].mean(axis=0)
                binned_centers.append(center)
        
        binned_centers = numpy.array(binned_centers)

        # # 5. Fit the Spline to the centers
        # # Smoothing 's' can be lower here because the binning already cleaned the noise
        tck, u = splprep(binned_centers.T, s=3.5, k=2)
        
        # # 6. Extend the object
        # # Evaluating from 0.0 to 1+extension extends the centerline
        smooth_centerline = numpy.array(splev(numpy.linspace(-extension_factor, 1.0 , 100), tck)).T

        if len(target_point)==0:
            return smooth_centerline
        
        if len(target_point)==1:
            smooth_centerline_extended = fit_target_constrained_spline(binned_centers, target_point[0], order, extension_factor)
            return smooth_centerline, smooth_centerline_extended
        
        if len(target_point)==2:
            smooth_centerline_extended = fit_dual_constrained_spline(binned_centers, target_point[0], target_point[1], order, extension_factor )
            return smooth_centerline, smooth_centerline_extended

        if len(target_point)==3:
            smooth_centerline_extended = fit_3_constrained_spline(binned_centers, target_point[0], target_point[1], target_point[2], order, extension_factor )
            return smooth_centerline, smooth_centerline_extended    


## Functions:
### Get insertion points ~ section curve / fitted planes

In [9]:
def fit_plane_to_two_centerlines(centerline1, centerline2):
    # 1. Combine all points from both centerlines into one cloud
    points = numpy.vstack((centerline1, centerline2))
    
    # 2. Calculate the centroid (the average point)
    centroid = numpy.mean(points, axis=0)
    
    # 3. Center the points by subtracting the centroid
    centered_points = points - centroid
    
    # 4. Use Singular Value Decomposition
    _, _, vh = numpy.linalg.svd(centered_points)
    
    # 5. The normal vector 'n' is the last row of Vh 
    # (the direction with the least variance/spread)
    normal = vh[-1]
    
    return centroid, normal/numpy.linalg.norm(normal)

def find_plane_curve_intersections(section, section_curve, plane_centroid, plane_normal):
    intersections = []
    
    # Normalize plane normal just in case
    plane_normal = plane_normal / numpy.linalg.norm(plane_normal)
    
    vertices = section.vertices
    
    for start_idx, end_idx in section_curve:
        p1 = vertices[start_idx]
        p2 = vertices[end_idx]
        
        # Signed distances to the plane
        d1 = numpy.dot(p1 - plane_centroid, plane_normal)
        d2 = numpy.dot(p2 - plane_centroid, plane_normal)
        
        # Check if the segment crosses the plane
        if d1 * d2 < 0:
            # Linear interpolation to find the exact crossing point
            t = abs(d1) / (abs(d1) + abs(d2))
            intersect_pt = p1 + t * (p2 - p1)
            intersections.append(intersect_pt)
            
    return numpy.array(intersections)



In [10]:
extended_centerlines = []
centerlines = []
extended_centerlines_1 = []

order = 3
# target_point = point on the lense axis

for muscle_id in range(len(muslce_ids)):
    mask = muscles[muscle_id]
    ignored_nodes = 0.01

    centerline, centerline_ext_1 = fit_analytical_centerline(mask, [target_point], ignored_nodes, order, n_bins=10, extension_factor=0.15)
    
    centerlines.append(centerline)
    extended_centerlines.append(centerline_ext_1)


    export_centerline(centerline_ext_1, pixel_spacing[0], base + folder + "centerline_exttended_1_id_"+str(muslce_ids[muscle_id])+".vtp", [0,0,0])
    export_centerline(centerline, pixel_spacing[0], base + folder + "centerline_init_id_"+str(muslce_ids[muscle_id])+".vtp", [0,0,0])

print("-------------------------")

point_hor, normal_vector_hor = fit_plane_to_two_centerlines(extended_centerlines[pairs[0][0]], extended_centerlines[pairs[0][1]])
point_vert, normal_vector_vert = fit_plane_to_two_centerlines(extended_centerlines[pairs[1][0]], extended_centerlines[pairs[1][1]])

print("Check of angle  = ", numpy.dot(normal_vector_hor, normal_vector_vert))

# Plane 1 intersections
pts_plane_horizontal = find_plane_curve_intersections(section, section_curve, point_hor*pixel_spacing[0], normal_vector_hor)
# Plane 2 intersections
pts_plane_vertical = find_plane_curve_intersections(section, section_curve, point_vert*pixel_spacing[0], normal_vector_vert)


# Plane 1B intersections
pts_plane_B_horizontal = find_plane_curve_intersections(section_B, section_curve_B, point_hor*pixel_spacing[0], normal_vector_hor)
# Plane 2B intersections
pts_plane_B_vertical = find_plane_curve_intersections(section_B, section_curve_B, point_vert*pixel_spacing[0], normal_vector_vert)

for id in range(len(muslce_ids)):

    print(id)

    if id in pairs[0]:
        print("muslce id ", muslce_ids[id],    "horizontal")

        targets_1 = pts_plane_horizontal/pixel_spacing[0]
        targets_1_B = pts_plane_B_horizontal/pixel_spacing[0]

    if id in pairs[1]:
        print("     muslce id ", muslce_ids[id],   " vertical")

        targets_1 = pts_plane_vertical/pixel_spacing[0]
        targets_1_B = pts_plane_B_vertical/pixel_spacing[0]

    # Extract coordinates of the current muscle mask
    mask = muscles[id]

    muscle_voxels = numpy.argwhere(mask) * pixel_spacing[0]
    tree = cKDTree(muscle_voxels)
    
    # Query distance to the chosen target points
    dist_to_target_1, _ = tree.query(targets_1 * pixel_spacing[0])
    dist_to_target_1_B, _ = tree.query(targets_1_B * pixel_spacing[0])
    
    print("        -> Distance to target_1: ", dist_to_target_1)
    print("        -> Distance to target_1_B: ", dist_to_target_1_B)

    target_1 = targets_1[numpy.argmin(dist_to_target_1)]
    target_1_B = targets_1_B[numpy.argmin(dist_to_target_1_B)]


    _, centerline_ext_2 = fit_analytical_centerline(mask, [target_1_B, target_1, target_point], ignored_nodes, order, n_bins=10, extension_factor=0.1)

    extended_centerlines_1.append(centerline_ext_2)


    export_centerline(centerline_ext_2, pixel_spacing[0], base + folder + "centerline_exttended_2_id_"+str(muslce_ids[id])+".vtp", [0,0,0])
    print("-------------------------")


reverse
constrained_points =  [[ 87.86727123 143.19950536  69.85078318]
 [ 77.99725275 137.5540293   71.68406593]
 [ 68.3591739  132.03501946  73.75516312]
 [ 60.06954887 125.11734694  75.35311493]
 [ 52.62008608 117.80114778  78.36700143]
 [ 46.12237237 109.37274775  80.75825826]
 [ 40.70101107  99.55849783  82.44487241]
 [ 35.12241055  89.67043315  83.31685499]
 [ 31.46343975  79.31204943  83.28939238]
 [ 67.90096257  34.53607004  84.90012487]]
reverse
constrained_points =  [[120.20923077 131.10666667  76.57538462]
 [119.2709599  121.51944107  77.87667072]
 [118.54614463 111.44546544  78.5669197 ]
 [118.33770807 101.60339309  79.79513444]
 [117.41531664  91.84948454  82.32430044]
 [116.64600378  81.98867212  83.77312775]
 [116.00493921  71.96200608  84.23328267]
 [114.20928931  61.95858982  84.78511472]
 [111.94178082  52.28561644  85.84863014]
 [ 67.90096257  34.53607004  84.90012487]]
reverse
constrained_points =  [[111.02972973 139.71351351  62.90990991]
 [106.66409465 129.7865226

# Part II

In [11]:
def get_local_frame(direction):
    """Calculates two vectors perpendicular to the spine direction."""
    # Pick a non-parallel vector to start the cross product
    # vec = numpy.array([1, 0, 0]) if abs(direction[0]) < 0.95 else numpy.array([0, 1, 0])

    vec_1 = numpy.array([1, 1, 0])
    vec = vec_1/numpy.linalg.norm(vec_1)

    v1 = numpy.cross(direction, vec)
    v1 /= numpy.linalg.norm(v1)
    v2 = numpy.cross(direction, v1)
    return v1, v2


## Extract profiles

In [12]:
def extract_radial_profiles_from_mask(centerline, mask, pixel_spacing, n_theta=64):
    """
    Step 1: Casts fixed radiating angular rays outward from each centerline node 
    to measure the exact distance (radius) to the mask boundary.
    """
    voxel_coords = numpy.argwhere(mask)
    if len(voxel_coords) == 0:
        return numpy.full((len(centerline), n_theta), numpy.nan)
        
    tree = cKDTree(voxel_coords)
    n_nodes = len(centerline)
    
    # Grid to hold our raw radii: rows = slices, columns = fixed angles
    radii_grid = numpy.zeros((n_nodes, n_theta))
    
    # Generate the fixed, matching angular target steps once
    theta_steps = numpy.linspace(-numpy.pi, numpy.pi, n_theta, endpoint=False)
    
    for idx in range(n_nodes):
        p = centerline[idx]
        
        # Calculate local tracking frame
        direction = centerline[min(idx+1, n_nodes-1)] - centerline[max(idx-1, 0)]
        norm_dir = numpy.linalg.norm(direction)
        if norm_dir == 0:
            radii_grid[idx, :] = numpy.nan
            continue
        direction /= norm_dir
        v1, v2 = get_local_frame(direction)
        
        # Broad neighborhood check
        close_idx = tree.query_ball_point(p, r=60)
        if len(close_idx) < 10:
            radii_grid[idx, :] = numpy.nan # Outside the mask volume
            continue
            
        local_voxels = voxel_coords[close_idx] - p
        dist_plane = numpy.dot(local_voxels, direction)
        slab = numpy.abs(dist_plane) < 1.0
        local_voxels = local_voxels[slab]
        
        if len(local_voxels) < 10:
            radii_grid[idx, :] = numpy.nan
            continue
            
        # Project local voxels onto 2D plane
        x = local_voxels @ v1
        y = local_voxels @ v2
        
        # Calculate polar coordinates of the raw slice voxels
        vox_theta = numpy.arctan2(y, x)
        vox_r = numpy.sqrt(x**2 + y**2)
        
        # Map the voxel distances onto our fixed angular target steps
        for t_idx, target_theta in enumerate(theta_steps):
            # Find boundary voxels near this specific ray angle (e.g., within 6 degrees)
            angle_diff = numpy.abs(numpy.arctan2(numpy.sin(vox_theta - target_theta), numpy.cos(vox_theta - target_theta)))
            angular_slice_mask = angle_diff < (2*numpy.pi / n_theta)
            
            if numpy.any(angular_slice_mask):
                # The maximum radius along this ray represents the outer boundary wall
                radii_grid[idx, t_idx] = numpy.max(vox_r[angular_slice_mask])
            else:
                radii_grid[idx, t_idx] = numpy.nan
                
    return radii_grid


def smooth_and_extrapolate_radii(radii_grid, n_output_extended, smoothing=15.0):
    """
    PASS 1: Longitudinal 1st-order (linear) spline extrapolation down each angle.
    PASS 2: In-slice cyclic wrapping filters to polish the 2D cross-sections.
    """
    n_trimmed, n_theta = radii_grid.shape
    
    # Initialize the output grid to fit the full extended target length
    smooth_profiles = numpy.zeros((n_output_extended, n_theta))

    # =========================================================================
    # PASS 1: LONGITUDINAL SMOOTHING & LINEAR EXTRAPOLATION (Across-Slice First)
    # =========================================================================
    u_real = numpy.arange(n_trimmed)
    u_evaluation = numpy.arange(n_output_extended)
    
    for t_idx in range(n_theta):
        column = radii_grid[:, t_idx]
        valid_mask = ~numpy.isnan(column)
        
        # Repair internal gaps within this angular column before fitting
        if numpy.any(valid_mask) and numpy.sum(valid_mask) >= 3:
            # Linear spline (k=1) to project raw trajectory straight into extension zone
            rib_spline = UnivariateSpline(
                u_real[valid_mask], 
                column[valid_mask], 
                k=1, 
                s=smoothing, 
                ext=0
            )
            extrapolated_radii = rib_spline(u_evaluation)
            smooth_profiles[:, t_idx] = numpy.clip(extrapolated_radii, 0.1, None)
        else:
            smooth_profiles[:, t_idx] = 1.0

    # Apply a quick longitudinal filter to stabilize the tracking strings
    for t_idx in range(n_theta):
        smooth_profiles[:, t_idx] = uniform_filter1d(
            smooth_profiles[:, t_idx], 
            size=5, 
            mode='nearest'
        )

    # =========================================================================
    # PASS 2: IN-SLICE PERIMETER SMOOTHING (In-Slice Second)
    # =========================================================================
    # Now that the full grid (up to n_output_extended) is populated, we loop
    # through EVERY slice and polish its circular contour perimeter.
    for idx in range(n_output_extended):
        row_series = smooth_profiles[idx, :].copy()
        
        # Failsafe check for empty rows
        if numpy.all(numpy.isnan(row_series)):
            continue
            
        # Clean up any leftover interpolation gaps around the loop
        nans = numpy.isnan(row_series)
        if numpy.any(nans):
            xp = numpy.where(~nans)[0]
            fp = row_series[~nans]
            row_series[nans] = numpy.interp(numpy.where(nans)[0], xp, fp, period=n_theta)
            
        # Cyclic wrapping filters erase any remaining jagged voxel steps
        filtered_row = median_filter(row_series, size=5, mode='wrap')
        smooth_profiles[idx, :] = uniform_filter1d(filtered_row, size=5, mode='wrap')
        
    return smooth_profiles



## Generate mesh

In [13]:
def generate_radial_mesh(
    centerline, 
    radii_grid, 
    pixel_spacing_val,
    insertion_idx=None,
    taper_start_len=12,
    taper_end_len=12
):
    print("Centerline shape:", centerline.shape)
    print("Radii grid shape:", radii_grid.shape)

    n_sections, n_theta = radii_grid.shape

    if insertion_idx is None:
        insertion_idx = n_sections

    n_sections = min(n_sections, insertion_idx)
    vertices = []
    
    theta_steps = numpy.linspace(-numpy.pi, numpy.pi, n_theta, endpoint=False)
    
    start_point = centerline[0]
    end_point = centerline[n_sections - 1]
    
    v1_prev = None
    # THE CRITICAL ADDITION: Accumulate the total twist angle continuously
    cumulative_twist = 0.0

    # -------------------------------------------------------------------------
    # 1. Slice Ring Geometry Mapping with Cumulative Realignment
    # -------------------------------------------------------------------------
    for i in range(n_sections):
        p_node = centerline[i]
        
        # Calculate local orientation frames natively 
        direction = centerline[min(i + 1, n_sections - 1)] - centerline[max(i - 1, 0)]
        norm_dir = numpy.linalg.norm(direction)
        direction = direction / norm_dir if norm_dir > 0 else numpy.array([1.0, 0.0, 0.0])
        v1, v2 = get_local_frame(direction)
        
        if v1_prev is not None:
            proj_x = numpy.dot(v1_prev, v1)
            proj_y = numpy.dot(v1_prev, v2)
            
            # Find the raw twist step between i-1 and i
            step_twist = numpy.arctan2(proj_y, proj_x)
            step_twist = (step_twist + numpy.pi) % (2.0 * numpy.pi) - numpy.pi

            # ONLY accumulate if it's an artificial mathematical jump (e.g., > 30 degrees)
            # This leaves the normal, correct global alignment completely untouched.
            if abs(step_twist) > 0.5:
                cumulative_twist += step_twist
            
        # Apply the total accumulated correction to this slice
        slice_theta = theta_steps # - cumulative_twist
            
        # Store the current frame to guide the next slice transition
        v1_prev = v1
        
        r_slice = radii_grid[i, :].copy()
        
        # =========================================================================
        # PROXIMAL SMOOTH DOME (Muscle Origin / Start)
        # =========================================================================
        if i < taper_start_len:
            t = i / max(taper_start_len - 1, 1)   
            scale = numpy.sqrt(t * (2.0 - t))     
            
            p_node = (1.0 - t) * start_point + t * p_node
            r_slice = r_slice * scale
            
        # =========================================================================
        # DISTAL SMOOTH DOME (Eye Wall Insertion / End)
        # =========================================================================
        if i >= n_sections - taper_end_len:
            t = (i - (n_sections - taper_end_len)) / max(taper_end_len - 1, 1) 
            scale = numpy.sqrt((1.0 - t) * (2.0 - (1.0 - t)))
            
            p_node = (1.0 - t) * p_node + t * end_point
            r_slice = r_slice * scale
            
        r_slice = numpy.maximum(r_slice, 0.05)
        
        # Build the ring using the adjusted slice_theta steps
        ring = (
            p_node[None, :]
            + r_slice[:, None] * numpy.cos(slice_theta)[:, None] * v1[None, :]
            + r_slice[:, None] * numpy.sin(slice_theta)[:, None] * v2[None, :]
        )
        vertices.append(ring)
        
    vertices = numpy.vstack(vertices)
    
    # -------------------------------------------------------------------------
    # 2. Append Explicit Singular Tip Vertices for Cap Enclosures
    # -------------------------------------------------------------------------
    start_tip_idx = len(vertices)
    vertices = numpy.vstack([vertices, start_point[None, :]])
    
    end_tip_idx = len(vertices)
    vertices = numpy.vstack([vertices, end_point[None, :]])
    
    # -------------------------------------------------------------------------
    # 3. Surface Topology Mapping (Closed Cap Enclosure)
    # -------------------------------------------------------------------------
    faces = []
    
    # A. Close Proximal Cap (Start Point)
    for j in range(n_theta):
        j2 = (j + 1) % n_theta
        faces.append([start_tip_idx, j2, j])
        
    # B. Construct Continuous Central Tube Quads
    for i in range(n_sections - 1):
        base0 = i * n_theta
        base1 = (i + 1) * n_theta
        
        for j in range(n_theta):
            j2 = (j + 1) % n_theta
            
            v00 = base0 + j
            v01 = base0 + j2
            v10 = base1 + j
            v11 = base1 + j2
            
            faces.append([v00, v10, v11])
            faces.append([v00, v11, v01])
            
    # C. Close Distal Cap (End Point)
    final_ring_base = (n_sections - 1) * n_theta
    for j in range(n_theta):
        j2 = (j + 1) % n_theta
        faces.append([final_ring_base + j, end_tip_idx, final_ring_base + j2])
            
    # -------------------------------------------------------------------------
    # 4. Compile and Export Clean Simulation Mesh Asset
    # -------------------------------------------------------------------------
    mesh = trimesh.Trimesh(
        vertices=vertices * pixel_spacing_val,
        faces=numpy.asarray(faces),
        process=False
    )
    
    return mesh

## Find intersection of the extended centerline and the eyeball


In [14]:
def find_eye_intersection_index(centerline_extended, mask_eye):
    """
    Returns the index along the extended centerline that intersects the eye.
    """
    for idx, point in enumerate(centerline_extended):
        z, y, x = numpy.round(point).astype(int)
        
        # Check bounds
        if 0 <= z < mask_eye.shape[0] and 0 <= y < mask_eye.shape[1] and 0 <= x < mask_eye.shape[2]:
            if mask_eye[z, y, x] == 1:
                return idx  
                
    return len(centerline_extended) - 1


## Meshin loop init

In [15]:
n_theta_points = 50
smoothing_factor = 5
spline_order = 3

for muscle_id in range(len(muslce_ids)):
    print("Running Simplified Radial Pipeline on Muscle:", muslce_ids[muscle_id])


    eye_intersection_idx = find_eye_intersection_index(
    extended_centerlines_1[muscle_id],
    mask_eye
    )

    short_centerline = centerlines[muscle_id]            # Original unextended path
    full_centerline = extended_centerlines_1[muscle_id]  # Extended path going to eye
    centerline_trimmed = extended_centerlines_1[muscle_id][:eye_intersection_idx+1]


   # ------------------------------------------------------------------------
    # 2. Step 1: Run the raw geometric radar sweep
    # -------------------------------------------------------------------------
    # This samples the mask data along the original short centerline length
    radii_raw = extract_radial_profiles_from_mask(
        centerline=short_centerline,
        mask=muscles[muscle_id],
        pixel_spacing=pixel_spacing,
        n_theta=n_theta_points
    )
    

    # -------------------------------------------------------------------------
    # 3. Steps 2 & 3: Smooth and Extrapolate to the Target Size
    radii_processed = smooth_and_extrapolate_radii(
        radii_grid=radii_raw,
        n_output_extended=len(short_centerline),
        smoothing= smoothing_factor
    )


    # -------------------------------------------------------------------------
    # 4. Step 4: Generate the surface mesh with biological end-tapers
    # -------------------------------------------------------------------------
    # We pass the full_centerline and set insertion_idx to match the eye boundary
    mesh = generate_radial_mesh(
        centerline=short_centerline,
        radii_grid=radii_processed,
        pixel_spacing_val=pixel_spacing[0],
        insertion_idx=eye_intersection_idx + 1,  # Clips rendering at eye wall
        taper_start_len=15,                      # Collapses origin smoothly
        taper_end_len=0                         # Collapses insertion smoothly
    )
    
    mesh.export(export_name + "_RADIAL_id_" + str(muslce_ids[muscle_id]) + ".stl")

Running Simplified Radial Pipeline on Muscle: 4
Centerline shape: (100, 3)
Radii grid shape: (100, 50)
Running Simplified Radial Pipeline on Muscle: 5
Centerline shape: (100, 3)
Radii grid shape: (100, 50)
Running Simplified Radial Pipeline on Muscle: 6
Centerline shape: (100, 3)
Radii grid shape: (100, 50)
Running Simplified Radial Pipeline on Muscle: 7
Centerline shape: (100, 3)
Radii grid shape: (100, 50)


## Contact surface prep

## Elliptic contact surface pt 1

In [16]:
def compute_local_tangent_from_section(section_2d_path, contact_point_mm):
    """
    Computes the normalized tangent vector using the topological 
    line entities of the trimesh section.
    """
    vertices = section_2d_path.vertices
    
    # 1. Find the entity segment closest to the contact point
    min_dist = float('inf')
    best_tangent = None
    
    for entity in section_2d_path.entities:
        # Entities contain ordered point indices, e.g. [idx_a, idx_b]
        points = entity.points
        for i in range(len(points) - 1):
            pA = vertices[points[i]]
            pB = vertices[points[i + 1]]
            
            # Midpoint of segment
            seg_mid = (pA + pB) / 2.0
            dist = numpy.linalg.norm(seg_mid - contact_point_mm)
            
            if dist < min_dist:
                min_dist = dist
                vec = pB - pA
                norm_vec = numpy.linalg.norm(vec)
                if norm_vec > 1e-8:
                    best_tangent = vec / norm_vec

    if best_tangent is None:
        raise ValueError("Could not extract a valid tangent from the section curve.")

    return best_tangent

## Elliptic contact surface pt 2

In [17]:
# def get_eyeball_elliptic_contact_surface(
#     eyeball_stl_path, 
#     centerline_extended, 
#     eye_intersection_idx, 
#     pixel_spacing, 
#     curve_tangent,             # Direction along section curve B
#     semi_major_mm=3.0,         # Ellipse length along tangent
#     semi_minor_mm=1.5,         # Ellipse width perpendicular to tangent
#     n_theta=50, 
#     output_vtk_path="eyeball_contact.vtk"
# ):
#     """
#     Extracts an elliptic contact patch aligned with a local path tangent direction.
#     """
#     # Load the eyeball mesh
#     mesh_trimesh = trimesh.load(eyeball_stl_path)
#     vertices = mesh_trimesh.vertices
#     faces = mesh_trimesh.faces
    
#     # 1. Contact point setup
#     contact_voxel_coord = centerline_extended[eye_intersection_idx]
#     contact_point_mm = contact_voxel_coord * pixel_spacing
    
#     mesh_tree = cKDTree(vertices)
#     dist, closest_vertex_idx = mesh_tree.query(contact_point_mm)
#     mesh_contact_point = vertices[closest_vertex_idx]
    
#     # -------------------------------------------------------------------------
#     # 1. Create Local Coordinate System (u, v, n) aligned with Section Tangent
#     # -------------------------------------------------------------------------
#     # u: Major axis along the curve tangent
#     u_axis = curve_tangent / numpy.linalg.norm(curve_tangent)
    
#     # Compute surface normal at contact point using vertex normals
#     surface_normal = mesh_trimesh.vertex_normals[closest_vertex_idx]
#     surface_normal = surface_normal/numpy.linalg.norm(surface_normal)
    
#     # v: Minor axis perpendicular to tangent and surface normal
#     v_axis = numpy.cross(surface_normal, u_axis)
#     v_axis /= numpy.linalg.norm(v_axis)
    
#     # Re-orthonormalize u_axis to guarantee pure perpendicular axes
#     u_axis = numpy.cross(v_axis, surface_normal)
#     u_axis /= numpy.linalg.norm(u_axis)

#     # -------------------------------------------------------------------------
#     # 2. Extract Elliptic Disc Sub-Mesh Patch
#     # -------------------------------------------------------------------------
#     face_centers = mesh_trimesh.triangles_center
    
#     # Fast spherical bounding query using max semi-axis radius
#     face_tree = cKDTree(face_centers)
#     candidate_indices = face_tree.query_ball_point(mesh_contact_point, r=max(semi_major_mm, semi_minor_mm) * 1.2)
    
#     # Filter candidates via elliptic metric: (x_local / a)^2 + (y_local / b)^2 <= 1
#     patch_face_indices = []
#     for f_idx in candidate_indices:
#         rel_pos = face_centers[f_idx] - mesh_contact_point
        
#         # Project onto local 2D tangent plane (u, v)
#         dist_u = numpy.dot(rel_pos, u_axis)
#         dist_v = numpy.dot(rel_pos, v_axis)
        
#         ellipse_metric = (dist_u / semi_major_mm)**2 + (dist_v / semi_minor_mm)**2
#         if ellipse_metric <= 1.0:
#             patch_face_indices.append(f_idx)
            
#     if len(patch_face_indices) < 3:
#         raise ValueError("Ellipse semi-axes are too small; no mesh faces captured.")
        
#     patch_mesh = mesh_trimesh.submesh([patch_face_indices], append=True)

#     # -------------------------------------------------------------------------
#     # 3. Boundary Extraction (Unique Edges)
#     # -------------------------------------------------------------------------
#     edges_sorted = patch_mesh.edges_sorted
#     unique_edge_indices = trimesh.grouping.group_rows(edges_sorted, require_count=1)
#     boundary_edges = patch_mesh.edges[unique_edge_indices]

#     # -------------------------------------------------------------------------
#     # 4. Adjacency Loop Walker
#     # -------------------------------------------------------------------------
#     adjacency = {}
#     for u, v in boundary_edges:
#         adjacency.setdefault(u, []).append(v)
#         adjacency.setdefault(v, []).append(u)
        
#     start_node = boundary_edges[0][0]
#     ordered_patch_vertex_indices = [start_node]
    
#     current_node = adjacency[start_node][0]
#     prev_node = start_node
    
#     while current_node != start_node:
#         ordered_patch_vertex_indices.append(current_node)
#         neighbors = adjacency[current_node]
#         next_node = neighbors[0] if neighbors[0] != prev_node else neighbors[1]
#         prev_node = current_node
#         current_node = next_node
        
#     raw_rim_vertices = patch_mesh.vertices[ordered_patch_vertex_indices]

#     # -------------------------------------------------------------------------
#     # 5. Angular Resampling aligned with Local Ellipse Frame
#     # -------------------------------------------------------------------------
#     rel_rim = raw_rim_vertices - mesh_contact_point
#     rim_angles = numpy.arctan2(rel_rim @ v_axis, rel_rim @ u_axis)
    
#     sort_idx = numpy.argsort(rim_angles)
#     sorted_rim_angles = rim_angles[sort_idx]
#     sorted_rim_vertices = raw_rim_vertices[sort_idx]
    
#     target_angles = numpy.linspace(-numpy.pi, numpy.pi, n_theta, endpoint=False)
#     ordered_contour_vertices = numpy.zeros((n_theta, 3))
    
#     for i in range(3):
#         ordered_contour_vertices[:, i] = numpy.interp(
#             target_angles, sorted_rim_angles, sorted_rim_vertices[:, i], period=2 * numpy.pi
#         )

#     # -------------------------------------------------------------------------
#     # 6. Export Attributed VTK Data
#     # -------------------------------------------------------------------------
#     nv = faces.shape[0]
#     padding = numpy.ones((nv, 1), dtype=int) * 3
#     pv_faces = numpy.hstack((padding, faces)).ravel()
#     grid = pv.PolyData(vertices, pv_faces)
    
#     global_rim_indices = []
#     global_tree = cKDTree(vertices)
#     for pt in ordered_contour_vertices:
#         _, glob_idx = global_tree.query(pt)
#         global_rim_indices.append(glob_idx)
        
#     global_patch_v_indices = numpy.unique(faces[patch_face_indices].ravel())
    
#     node_registration = numpy.zeros(grid.n_points, dtype=numpy.int32)
#     node_registration[global_patch_v_indices] = 2  # Elliptic patch interior
#     node_registration[global_rim_indices] = 3      # Boundary rim
#     node_registration[closest_vertex_idx] = 1     # Contact center point
    
#     grid.point_data["NodeRegistration"] = node_registration
#     grid.save(output_vtk_path)
    
#     print(f"--> [SUCCESS] Saved Elliptic VTK mesh to: {output_vtk_path}")
#     return mesh_contact_point, ordered_contour_vertices

In [18]:
def get_eyeball_elliptic_contact_surface(
    eyeball_stl_path, 
    centerline_extended, 
    eye_intersection_idx, 
    pixel_spacing, 
    curve_tangent,             # Direction along section curve B
    semi_major_mm=3.0,         # Ellipse length along tangent
    semi_minor_mm=1.5,         # Ellipse width perpendicular to tangent
    n_theta=50, 
    offset_mm=1.0,             # <--- NEW: Outward gap distance from eyeball surface
    output_vtk_path="eyeball_contact.vtk"
):
    """
    Extracts an elliptic contact patch aligned with a local path tangent direction,
    and offsets it outward from the eyeball surface to create a clearance gap.
    """
    # Load the eyeball mesh
    mesh_trimesh = trimesh.load(eyeball_stl_path)
    vertices = mesh_trimesh.vertices
    faces = mesh_trimesh.faces
    
    # 1. Contact point setup
    contact_voxel_coord = centerline_extended[eye_intersection_idx]
    contact_point_mm = contact_voxel_coord * pixel_spacing
    
    mesh_tree = cKDTree(vertices)
    dist, closest_vertex_idx = mesh_tree.query(contact_point_mm)
    mesh_contact_point = vertices[closest_vertex_idx]
    
    # -------------------------------------------------------------------------
    # 1. Create Local Coordinate System (u, v, n) aligned with Section Tangent
    # -------------------------------------------------------------------------
    u_axis = curve_tangent / numpy.linalg.norm(curve_tangent)
    
    # Compute surface normal at contact point using vertex normals
    surface_normal = mesh_trimesh.vertex_normals[closest_vertex_idx]
    surface_normal = surface_normal / numpy.linalg.norm(surface_normal)
    
    # v: Minor axis perpendicular to tangent and surface normal
    v_axis = numpy.cross(surface_normal, u_axis)
    v_axis /= numpy.linalg.norm(v_axis)
    
    # Re-orthonormalize u_axis to guarantee pure perpendicular axes
    u_axis = numpy.cross(v_axis, surface_normal)
    u_axis /= numpy.linalg.norm(u_axis)

    # -------------------------------------------------------------------------
    # 2. Extract Elliptic Disc Sub-Mesh Patch
    # -------------------------------------------------------------------------
    face_centers = mesh_trimesh.triangles_center
    
    face_tree = cKDTree(face_centers)
    candidate_indices = face_tree.query_ball_point(mesh_contact_point, r=max(semi_major_mm, semi_minor_mm) * 1.2)
    
    patch_face_indices = []
    for f_idx in candidate_indices:
        rel_pos = face_centers[f_idx] - mesh_contact_point
        
        dist_u = numpy.dot(rel_pos, u_axis)
        dist_v = numpy.dot(rel_pos, v_axis)
        
        ellipse_metric = (dist_u / semi_major_mm)**2 + (dist_v / semi_minor_mm)**2
        if ellipse_metric <= 1.0:
            patch_face_indices.append(f_idx)
            
    if len(patch_face_indices) < 3:
        raise ValueError("Ellipse semi-axes are too small; no mesh faces captured.")
        
    patch_mesh = mesh_trimesh.submesh([patch_face_indices], append=True)

    # -------------------------------------------------------------------------
    # 3. Boundary Extraction (Unique Edges)
    # -------------------------------------------------------------------------
    edges_sorted = patch_mesh.edges_sorted
    unique_edge_indices = trimesh.grouping.group_rows(edges_sorted, require_count=1)
    boundary_edges = patch_mesh.edges[unique_edge_indices]

    # -------------------------------------------------------------------------
    # 4. Adjacency Loop Walker
    # -------------------------------------------------------------------------
    adjacency = {}
    for u, v in boundary_edges:
        adjacency.setdefault(u, []).append(v)
        adjacency.setdefault(v, []).append(u)
        
    start_node = boundary_edges[0][0]
    ordered_patch_vertex_indices = [start_node]
    
    current_node = adjacency[start_node][0]
    prev_node = start_node
    
    while current_node != start_node:
        ordered_patch_vertex_indices.append(current_node)
        neighbors = adjacency[current_node]
        next_node = neighbors[0] if neighbors[0] != prev_node else neighbors[1]
        prev_node = current_node
        current_node = next_node
        
    raw_rim_vertices = patch_mesh.vertices[ordered_patch_vertex_indices]

    # -------------------------------------------------------------------------
    # 5. Angular Resampling aligned with Local Ellipse Frame
    # -------------------------------------------------------------------------
    rel_rim = raw_rim_vertices - mesh_contact_point
    rim_angles = numpy.arctan2(rel_rim @ v_axis, rel_rim @ u_axis)
    
    sort_idx = numpy.argsort(rim_angles)
    sorted_rim_angles = rim_angles[sort_idx]
    sorted_rim_vertices = raw_rim_vertices[sort_idx]
    
    target_angles = numpy.linspace(-numpy.pi, numpy.pi, n_theta, endpoint=False)
    ordered_contour_vertices = numpy.zeros((n_theta, 3))
    
    for i in range(3):
        ordered_contour_vertices[:, i] = numpy.interp(
            target_angles, sorted_rim_angles, sorted_rim_vertices[:, i], period=2 * numpy.pi
        )

    # -------------------------------------------------------------------------
    # 6. APPLY OUTWARD OFFSET (FOR MUSCLE ATTACHMENT)
    # -------------------------------------------------------------------------
    if offset_mm > 0:
        # Option A: Smooth directional shift using local vertex normals (preserves local surface curvature)
        # Interpolate vertex normals for the resampled rim points
        rim_tree = cKDTree(patch_mesh.vertices)
        _, rim_to_patch_indices = rim_tree.query(ordered_contour_vertices)
        contour_normals = patch_mesh.vertex_normals[rim_to_patch_indices]
        
        # Normalize vectors just in case
        contour_normals /= numpy.linalg.norm(contour_normals, axis=1, keepdims=True)
        
        # Push contour points outward along their local surface normals
        muscle_contour_vertices = ordered_contour_vertices + contour_normals * offset_mm
        
        # Push central point outward along main surface normal
        muscle_contact_point = mesh_contact_point + surface_normal * offset_mm
    else:
        muscle_contour_vertices = ordered_contour_vertices
        muscle_contact_point = mesh_contact_point

    # -------------------------------------------------------------------------
    # 7. Export Attributed VTK Data
    # -------------------------------------------------------------------------
    nv = faces.shape[0]
    padding = numpy.ones((nv, 1), dtype=int) * 3
    pv_faces = numpy.hstack((padding, faces)).ravel()
    grid = pv.PolyData(vertices, pv_faces)
    
    global_rim_indices = []
    global_tree = cKDTree(vertices)
    for pt in ordered_contour_vertices:
        _, glob_idx = global_tree.query(pt)
        global_rim_indices.append(glob_idx)
        
    global_patch_v_indices = numpy.unique(faces[patch_face_indices].ravel())
    
    node_registration = numpy.zeros(grid.n_points, dtype=numpy.int32)
    node_registration[global_patch_v_indices] = 2  # Elliptic patch interior
    node_registration[global_rim_indices] = 3      # Boundary rim
    node_registration[closest_vertex_idx] = 1     # Contact center point
    
    grid.point_data["NodeRegistration"] = node_registration
    grid.save(output_vtk_path)
    
    print(f"--> [SUCCESS] Saved Elliptic VTK mesh to: {output_vtk_path}")
    
    # Return the OFFSETTED points so the muscle mesh uses them
    return muscle_contact_point, muscle_contour_vertices

## Contact surface on the eyeball mesh

In [19]:
def get_eyeball_elliptic_contact_surface(
    eyeball_stl_path, 
    centerline_extended, 
    eye_intersection_idx, 
    pixel_spacing, 
    curve_tangent,             # Direction along section curve B
    semi_major_mm=2.0,         # Length along section curve
    semi_minor_mm=2.0,         # Width perpendicular to section curve
    n_theta=50, 
    output_vtk_path="eyeball_contact.vtk"
):
    # Load mesh
    mesh_trimesh = trimesh.load(eyeball_stl_path)
    vertices = mesh_trimesh.vertices
    faces = mesh_trimesh.faces
    
    # 1. Contact point setup
    contact_voxel_coord = centerline_extended[eye_intersection_idx]
    contact_point_mm = contact_voxel_coord * pixel_spacing
    
    mesh_tree = cKDTree(vertices)
    dist, closest_vertex_idx = mesh_tree.query(contact_point_mm)
    mesh_contact_point = vertices[closest_vertex_idx]
    
    # -------------------------------------------------------------------------
    # 1. Local Tangent Frame for Elliptic Spatial Filter
    # -------------------------------------------------------------------------
    u_axis = curve_tangent / numpy.linalg.norm(curve_tangent)
    surface_normal = mesh_trimesh.vertex_normals[closest_vertex_idx]
    surface_normal = surface_normal/numpy.linalg.norm(surface_normal)
    
    v_axis = numpy.cross(surface_normal, u_axis)
    v_axis /= numpy.linalg.norm(v_axis)
    u_axis = numpy.cross(v_axis, surface_normal)
    u_axis /= numpy.linalg.norm(u_axis)

    # -------------------------------------------------------------------------
    # 2. Extract Elliptic Submesh Patch
    # -------------------------------------------------------------------------
    face_centers = mesh_trimesh.triangles_center
    face_tree = cKDTree(face_centers)
    candidate_indices = face_tree.query_ball_point(mesh_contact_point, r=max(semi_major_mm, semi_minor_mm) * 1.2)
    
    patch_face_indices = []
    for f_idx in candidate_indices:
        rel_pos = face_centers[f_idx] - mesh_contact_point
        dist_u = numpy.dot(rel_pos, u_axis)
        dist_v = numpy.dot(rel_pos, v_axis)
        
        if (dist_u / semi_major_mm)**2 + (dist_v / semi_minor_mm)**2 <= 1.0:
            patch_face_indices.append(f_idx)
            
    if len(patch_face_indices) < 3:
        raise ValueError("Ellipse semi-axes are too small; no mesh faces captured.")
        
    patch_mesh = mesh_trimesh.submesh([patch_face_indices], append=True)

    # -------------------------------------------------------------------------
    # 3. Boundary Extraction & Adjacency Loop Walk
    # -------------------------------------------------------------------------
    edges_sorted = patch_mesh.edges_sorted
    unique_edge_indices = trimesh.grouping.group_rows(edges_sorted, require_count=1)
    boundary_edges = patch_mesh.edges[unique_edge_indices]

    adjacency = {}
    for u, v in boundary_edges:
        adjacency.setdefault(u, []).append(v)
        adjacency.setdefault(v, []).append(u)
        
    start_node = boundary_edges[0][0]
    ordered_patch_vertex_indices = [start_node]
    
    current_node = adjacency[start_node][0]
    prev_node = start_node
    
    while current_node != start_node:
        ordered_patch_vertex_indices.append(current_node)
        neighbors = adjacency[current_node]
        next_node = neighbors[0] if neighbors[0] != prev_node else neighbors[1]
        prev_node = current_node
        current_node = next_node
        
    raw_rim_vertices = patch_mesh.vertices[ordered_patch_vertex_indices]

    # -------------------------------------------------------------------------
    # 4. Angular Synchronization (EXACT MATCH TO YOUR ORIGINAL CODE)
    # -------------------------------------------------------------------------
    # We use the centerline tangent (same frame reference as original code)
    idx_next = min(eye_intersection_idx + 1, len(centerline_extended) - 1)
    idx_prev = max(eye_intersection_idx - 1, 0)
    centerline_tangent = (centerline_extended[idx_next] - centerline_extended[idx_prev]) * pixel_spacing
    centerline_tangent /= numpy.linalg.norm(centerline_tangent)
    
    if numpy.abs(centerline_tangent[0]) < 0.9:
        v1 = numpy.cross(centerline_tangent, [1, 0, 0])
    else:
        v1 = numpy.cross(centerline_tangent, [0, 1, 0])
    v1 /= numpy.linalg.norm(v1)
    v2 = numpy.cross(centerline_tangent, v1)
    v2 /= numpy.linalg.norm(v2)
    
    # Project rim vertices into original (v1, v2) coordinate frame
    rel_rim = raw_rim_vertices - mesh_contact_point
    rim_angles = numpy.arctan2(rel_rim @ v2, rel_rim @ v1)
    
    sort_idx = numpy.argsort(rim_angles)
    sorted_rim_angles = rim_angles[sort_idx]
    sorted_rim_vertices = raw_rim_vertices[sort_idx]
    
    target_angles = numpy.linspace(-numpy.pi, numpy.pi, n_theta, endpoint=False)
    ordered_contour_vertices = numpy.zeros((n_theta, 3))
    
    for i in range(3):
        ordered_contour_vertices[:, i] = numpy.interp(
            target_angles, sorted_rim_angles, sorted_rim_vertices[:, i], period=2 * numpy.pi
        )

    # -------------------------------------------------------------------------
    # 5. Export Attributed VTK Data
    # -------------------------------------------------------------------------
    nv = faces.shape[0]
    padding = numpy.ones((nv, 1), dtype=int) * 3
    pv_faces = numpy.hstack((padding, faces)).ravel()
    grid = pv.PolyData(vertices, pv_faces)
    
    global_rim_indices = []
    global_tree = cKDTree(vertices)
    for pt in ordered_contour_vertices:
        _, glob_idx = global_tree.query(pt)
        global_rim_indices.append(glob_idx)
        
    global_patch_v_indices = numpy.unique(faces[patch_face_indices].ravel())
    
    node_registration = numpy.zeros(grid.n_points, dtype=numpy.int32)
    node_registration[global_patch_v_indices] = 2
    node_registration[global_rim_indices] = 3
    node_registration[closest_vertex_idx] = 1
    
    grid.point_data["NodeRegistration"] = node_registration
    grid.save(output_vtk_path)
    
    return mesh_contact_point, ordered_contour_vertices




## Set meshing parameters

In [20]:
n_theta_points = 55
smoothing_factor = 15
spline_order = 3

## Get contact points & surfaces

In [21]:
contact_centers = []
eye_surfaces = []
eye_intersection_IDs = []

for muscle_id in range(len(muslce_ids)):

    print("muslce : ", muslce_ids[muscle_id])
    
    eyeball_stl = f"{export_name}_surf_id_{str(eyeball_id)}_outer.stl"
    output_vtk = f"{export_name}_muscle_{muslce_ids[muscle_id]}_eye_contact.vtk"

    eye_intersection_idx = find_eye_intersection_index(
    extended_centerlines_1[muscle_id],
    mask_eye
    )
    eye_intersection_IDs.append(eye_intersection_idx)
    
    # # Original functioning version with circle
    # contact_center, eye_wall_ring = get_eyeball_contact_surface(
    #     eyeball_stl_path=eyeball_stl,
    #     centerline_extended=extended_centerlines_1[muscle_id],
    #     eye_intersection_idx=eye_intersection_idx,
    #     pixel_spacing=pixel_spacing[0],
    #     target_radius_mm=2.0, # Target width
    #     n_theta=n_theta_points,
    #     output_vtk_path=output_vtk
    # )

    # Elliptic contact surface
    # 2. Extract tangent direction at contact point
    contact_pt_mm = extended_centerlines_1[muscle_id][eye_intersection_idx] * pixel_spacing[0]
    curve_tangent = compute_local_tangent_from_section(section, contact_pt_mm)

    # 3. Pass tangent to elliptic contact surface generator
    contact_center, eye_wall_ring = get_eyeball_elliptic_contact_surface(
        eyeball_stl_path=eyeball_stl,
        centerline_extended=extended_centerlines_1[muscle_id],
        eye_intersection_idx=eye_intersection_idx,
        pixel_spacing=pixel_spacing[0],
        curve_tangent=curve_tangent,
        semi_major_mm=contact_axis_1,  # Length along section curve
        semi_minor_mm=contact_axis_2,  # Width perpendicular to section curve
        n_theta=n_theta_points,
        output_vtk_path=output_vtk
    )

    contact_centers.append(contact_center)
    eye_surfaces.append(eye_wall_ring)



muslce :  4
muslce :  5
muslce :  6
muslce :  7


## Meshing function

### OK / best so far

In [22]:
# import scipy.spatial
# import scipy.interpolate

# def get_local_frame(direction):
#     """
#     Original frame generator using fixed [1, 1, 0] reference vector.
#     """
#     vec_1 = numpy.array([1.0, 1.0, 0.0])
#     vec = vec_1 / numpy.linalg.norm(vec_1)

#     norm_dir = numpy.linalg.norm(direction)
#     t = direction / norm_dir if norm_dir > 1e-8 else numpy.array([1.0, 0.0, 0.0])

#     if numpy.abs(numpy.dot(t, vec)) > 0.99:
#         vec = numpy.array([0.0, 0.0, 1.0])

#     v1 = numpy.cross(t, vec)
#     norm_v1 = numpy.linalg.norm(v1)
#     if norm_v1 > 1e-8:
#         v1 /= norm_v1
#     else:
#         v1 = numpy.array([1.0, 0.0, 0.0])

#     v2 = numpy.cross(t, v1)
#     norm_v2 = numpy.linalg.norm(v2)
#     if norm_v2 > 1e-8:
#         v2 /= norm_v2

#     return v1, v2

# def smooth_centerline_segment(centerline, start_idx, smoothing_factor=0.5):
#     pts = centerline.copy()
#     if len(pts) - start_idx < 4:
#         return pts
        
#     transitional_pts = pts[start_idx:]
#     t = numpy.linspace(0, 1, len(transitional_pts))
    
#     try:
#         spl_x = scipy.interpolate.UnivariateSpline(t, transitional_pts[:, 0], s=smoothing_factor)
#         spl_y = scipy.interpolate.UnivariateSpline(t, transitional_pts[:, 1], s=smoothing_factor)
#         spl_z = scipy.interpolate.UnivariateSpline(t, transitional_pts[:, 2], s=smoothing_factor)
        
#         pts[start_idx:, 0] = spl_x(t)
#         pts[start_idx:, 1] = spl_y(t)
#         pts[start_idx:, 2] = spl_z(t)
#     except Exception:
#         pass
        
#     return pts

# def generate_blended_mesh(
#     full_centerline, 
#     short_centerline, 
#     short_radii_grid, 
#     eye_wall_ring, 
#     contact_center_mm, 
#     pixel_spacing_val, 
#     origin_center_mm=None,
#     spine_smoothing=5.0,
#     blend_power=2.5
# ):
#     """
#     Generates smooth muscle mesh using continuous arc-length parameter interpolation 
#     to eliminate periodic radius stair-stepping/jumps in Zone 1.
#     """
#     n_total_slices = len(full_centerline)             
#     n_real_slices, n_theta = short_radii_grid.shape    
#     vertices = []
    
#     theta_steps = numpy.linspace(-numpy.pi, numpy.pi, n_theta, endpoint=False)
    
#     eye_wall_ring_vox = eye_wall_ring / pixel_spacing_val
#     contact_center_vox = contact_center_mm / pixel_spacing_val
    
#     # -------------------------------------------------------------------------
#     # 1. ANCHOR MAPPING & CONTINUOUS PARAMETERIZATION
#     # -------------------------------------------------------------------------
#     full_tree = scipy.spatial.KDTree(full_centerline)
    
#     _, y_start_idx = full_tree.query(short_centerline[0])
#     _, y_end_idx = full_tree.query(short_centerline[-1])
    
#     smoothed_centerline = smooth_centerline_segment(full_centerline, y_end_idx, smoothing_factor=spine_smoothing)
    
#     if origin_center_mm is None:
#         origin_center_vox = smoothed_centerline[y_start_idx]
#     else:
#         origin_center_vox = origin_center_mm / pixel_spacing_val
    
#     taper_start_len = 12

#     # Build continuous interpolation for Zone 1 radii along arc-length
#     t_short = numpy.linspace(0, 1, n_real_slices)
#     radii_interpolator = scipy.interpolate.interp1d(
#         t_short, short_radii_grid, axis=0, kind='linear', fill_value="extrapolate"
#     )
    
#     # Tail profile continuously sampled at the end of Zone 1
#     r_tail = radii_interpolator(1.0)

#     # -------------------------------------------------------------------------
#     # 2. FRAME GENERATION (ZONE 1 & ZONE 2)
#     # -------------------------------------------------------------------------
#     all_v1 = []
#     all_v2 = []

#     # Zone 1: Pure native frames
#     for i in range(y_start_idx, y_end_idx + 1):
#         dir_vec = smoothed_centerline[min(i + 1, n_total_slices - 1)] - smoothed_centerline[max(i - 1, 0)]
#         v1, v2 = get_local_frame(dir_vec)
#         all_v1.append(v1)
#         all_v2.append(v2)

#     # Zone 2: Parallel Transport extension
#     for i in range(y_end_idx + 1, n_total_slices - 1):
#         dir_vec = smoothed_centerline[min(i + 1, n_total_slices - 1)] - smoothed_centerline[max(i - 1, 0)]
#         norm_dir = numpy.linalg.norm(dir_vec)
#         t_curr = dir_vec / norm_dir if norm_dir > 0 else numpy.array([1.0, 0.0, 0.0])

#         v1_prev = all_v1[-1]
#         v1_next = v1_prev - numpy.dot(v1_prev, t_curr) * t_curr
#         norm_v1 = numpy.linalg.norm(v1_next)
#         if norm_v1 > 1e-8:
#             v1_next /= norm_v1
#         else:
#             v1_next = v1_prev

#         v2_next = numpy.cross(t_curr, v1_next)
#         v2_next /= numpy.linalg.norm(v2_next)

#         all_v1.append(v1_next)
#         all_v2.append(v2_next)

#     # -------------------------------------------------------------------------
#     # 3. ALIGN EYE RING PHASE
#     # -------------------------------------------------------------------------
#     p_touchdown = smoothed_centerline[-1]
#     v1_end, v2_end = all_v1[-1], all_v2[-1]

#     raw_eye_offsets = eye_wall_ring_vox - p_touchdown[None, :]

#     x_proj = numpy.dot(raw_eye_offsets, v1_end)
#     y_proj = numpy.dot(raw_eye_offsets, v2_end)
    
#     eye_angles = numpy.arctan2(y_proj, x_proj)
#     eye_radii = numpy.sqrt(x_proj**2 + y_proj**2)

#     sort_idx = numpy.argsort(eye_angles)
#     sorted_angles = eye_angles[sort_idx]
#     sorted_radii = eye_radii[sort_idx]
    
#     sorted_angles_ext = numpy.concatenate([sorted_angles - 2*numpy.pi, sorted_angles, sorted_angles + 2*numpy.pi])
#     sorted_radii_ext = numpy.concatenate([sorted_radii, sorted_radii, sorted_radii])
    
#     r_eye_aligned = scipy.interpolate.interp1d(sorted_angles_ext, sorted_radii_ext, kind='linear')(theta_steps)

#     # -------------------------------------------------------------------------
#     # 4. BUILD GEOMETRY SLICE-BY-SLICE
#     # -------------------------------------------------------------------------
#     zone1_total_steps = max(y_end_idx - y_start_idx, 1)

#     for i in range(y_start_idx, n_total_slices):
#         local_i = i - y_start_idx
#         p_node = smoothed_centerline[i]
#         slice_theta = theta_steps 
        
#         v1_curr = all_v1[local_i] if local_i < len(all_v1) else v1_end
#         v2_curr = all_v2[local_i] if local_i < len(all_v2) else v2_end

#         # ZONE 1: CORE MUSCLE BODY (Continuous Interpolation)
#         if i <= y_end_idx:
#             # Normalized progression along Zone 1 [0.0 to 1.0]
#             t_zone1 = local_i / zone1_total_steps
#             r_slice = radii_interpolator(t_zone1)
            
#             start_point = smoothed_centerline[y_start_idx]
#             if local_i < taper_start_len:
#                 t_taper = local_i / max(taper_start_len - 1, 1)
#                 scale = numpy.sqrt(t_taper * (2.0 - t_taper))  
#                 p_node = (1.0 - t_taper) * start_point + t_taper * p_node
#                 r_slice = r_slice * scale
                
#             r_slice = numpy.maximum(r_slice, 0.05)
            
#             ring = (
#                 p_node[None, :]
#                 + r_slice[:, None] * numpy.cos(slice_theta)[:, None] * v1_curr[None, :]
#                 + r_slice[:, None] * numpy.sin(slice_theta)[:, None] * v2_curr[None, :]
#             )
#             vertices.append(ring)
            
#         # ZONE 2: TRANSITIONAL TUBE
#         elif i < n_total_slices - 1:
#             trans_idx = i - y_end_idx
#             gap_total_steps = (n_total_slices - 1) - y_end_idx
            
#             t_lin = trans_idx / max(gap_total_steps, 1)
#             t_blend = t_lin ** blend_power
            
#             r_curr = (1.0 - t_blend) * r_tail + t_blend * r_eye_aligned
            
#             ring = (
#                 p_node[None, :]
#                 + r_curr[:, None] * numpy.cos(slice_theta)[:, None] * v1_curr[None, :]
#                 + r_curr[:, None] * numpy.sin(slice_theta)[:, None] * v2_curr[None, :]
#             )
#             vertices.append(ring)
            
#         # ZONE 3: FINAL TOUCHDOWN SEAM
#         else:
#             vertices.append(eye_wall_ring_vox)
        
#     vertices = numpy.vstack(vertices)
    
#     # -------------------------------------------------------------------------
#     # 5. TOPOLOGY ASSEMBLY
#     # -------------------------------------------------------------------------
#     n_rendered_slices = n_total_slices - y_start_idx
    
#     start_tip_idx = len(vertices)
#     vertices = numpy.vstack([vertices, origin_center_vox[None, :]])
    
#     end_tip_idx = len(vertices)
#     vertices = numpy.vstack([vertices, contact_center_vox[None, :]])
    
#     faces = []
    
#     # Proximal Cap
#     for j in range(n_theta):
#         j2 = (j + 1) % n_theta
#         faces.append([start_tip_idx, j2, j])
        
#     # Main Tube Quads
#     for i in range(n_rendered_slices - 1):
#         base0 = i * n_theta
#         base1 = (i + 1) * n_theta
#         for j in range(n_theta):
#             j2 = (j + 1) % n_theta
#             faces.append([base0 + j, base1 + j, base1 + j2])
#             faces.append([base0 + j, base1 + j2, base0 + j2])
            
#     # Distal Cap
#     final_ring_base = (n_rendered_slices - 1) * n_theta
#     for j in range(n_theta):
#         j2 = (j + 1) % n_theta
#         faces.append([final_ring_base + j, end_tip_idx, final_ring_base + j2])
        
#     mesh = trimesh.Trimesh(
#         vertices=vertices * pixel_spacing_val,
#         faces=numpy.asarray(faces),
#         process=False
#     )
    
#     return mesh

In [23]:
def fast_fill_cuts_humphrey(mesh, iterations=10, alpha=0.1, beta=0.5):
    """
    Rapidly bridges thin cuts, pinches, and sharp concave creases 
    without translating or shrinking the surface mesh.
    
    :param mesh: Input trimesh object
    :param iterations: 5-15 passes is usually the sweet spot
    :param alpha: Neighborhood influence (higher = bridges wider cuts)
    :param beta: Volume preservation factor (0.5 maintains original scale)
    """
    repaired = mesh.copy()
    
    # Humphrey's smoothing operates directly in-place on vertex positions
    trimesh.smoothing.filter_humphrey(
        repaired, 
        alpha=alpha, 
        beta=beta, 
        iterations=iterations
    )
    
    return repaired

In [24]:
import numpy
import scipy.spatial
import scipy.interpolate
import trimesh

def get_local_frame(direction):
    """
    Original frame generator using fixed [1, 1, 0] reference vector.
    """
    vec_1 = numpy.array([1.0, 1.0, 0.0])
    vec = vec_1 / numpy.linalg.norm(vec_1)

    norm_dir = numpy.linalg.norm(direction)
    t = direction / norm_dir if norm_dir > 1e-8 else numpy.array([1.0, 0.0, 0.0])

    if numpy.abs(numpy.dot(t, vec)) > 0.99:
        vec = numpy.array([0.0, 0.0, 1.0])

    v1 = numpy.cross(t, vec)
    norm_v1 = numpy.linalg.norm(v1)
    if norm_v1 > 1e-8:
        v1 /= norm_v1
    else:
        v1 = numpy.array([1.0, 0.0, 0.0])

    v2 = numpy.cross(t, v1)
    norm_v2 = numpy.linalg.norm(v2)
    if norm_v2 > 1e-8:
        v2 /= norm_v2

    return v1, v2

def smooth_centerline_segment(centerline, start_idx, smoothing_factor=0.5):
    pts = centerline.copy()
    if len(pts) - start_idx < 4:
        return pts
        
    transitional_pts = pts[start_idx:]
    t = numpy.linspace(0, 1, len(transitional_pts))
    
    try:
        spl_x = scipy.interpolate.UnivariateSpline(t, transitional_pts[:, 0], s=smoothing_factor)
        spl_y = scipy.interpolate.UnivariateSpline(t, transitional_pts[:, 1], s=smoothing_factor)
        spl_z = scipy.interpolate.UnivariateSpline(t, transitional_pts[:, 2], s=smoothing_factor)
        
        pts[start_idx:, 0] = spl_x(t)
        pts[start_idx:, 1] = spl_y(t)
        pts[start_idx:, 2] = spl_z(t)
    except Exception:
        pass
        
    return pts

# def generate_blended_mesh(
#     full_centerline, 
#     short_centerline, 
#     short_radii_grid, 
#     eye_wall_ring, 
#     contact_center_mm, 
#     pixel_spacing_val, 
#     origin_center_mm=None,
#     spine_smoothing=5.0,
#     blend_power=2.5,
#     stride=1  # <--- Increasing this creates thicker layers (e.g., 2, 3, or 4)
# ):
#     """
#     Generates smooth muscle mesh using continuous arc-length parameter interpolation.
#     Includes slice downsampling via `stride` to create thicker layers and prevent
#     contour/plane intersections on bends.
#     """
#     n_real_slices, n_theta = short_radii_grid.shape    
#     vertices = []
    
#     theta_steps = numpy.linspace(-numpy.pi, numpy.pi, n_theta, endpoint=False)
    
#     eye_wall_ring_vox = eye_wall_ring / pixel_spacing_val
#     contact_center_vox = contact_center_mm / pixel_spacing_val
    
#     # -------------------------------------------------------------------------
#     # 1. ANCHOR MAPPING & CONTINUOUS PARAMETERIZATION
#     # -------------------------------------------------------------------------
#     full_tree = scipy.spatial.KDTree(full_centerline)
    
#     _, y_start_idx = full_tree.query(short_centerline[0])
#     _, y_end_idx = full_tree.query(short_centerline[-1])
    
#     smoothed_centerline = smooth_centerline_segment(full_centerline, y_end_idx, smoothing_factor=spine_smoothing)
    
#     # --- DOWNSAMPLE SLICES (Thicker Layers) ---
#     # Construct slice indices with `stride`, explicitly keeping key junction boundaries
#     zone1_indices = list(range(y_start_idx, y_end_idx, stride))
#     if y_end_idx not in zone1_indices:
#         zone1_indices.append(y_end_idx)

#     zone2_indices = list(range(y_end_idx + stride, len(smoothed_centerline) - 1, stride))
#     if (len(smoothed_centerline) - 1) not in zone2_indices:
#         zone2_indices.append(len(smoothed_centerline) - 1)

#     rendered_indices = zone1_indices + zone2_indices
#     n_rendered_slices = len(rendered_indices)
    
#     if origin_center_mm is None:
#         origin_center_vox = smoothed_centerline[y_start_idx]
#     else:
#         origin_center_vox = origin_center_mm / pixel_spacing_val
    
#     taper_start_len = max(int(12 / stride), 1)

#     # Build continuous interpolation for Zone 1 radii along arc-length
#     t_short = numpy.linspace(0, 1, n_real_slices)
#     radii_interpolator = scipy.interpolate.interp1d(
#         t_short, short_radii_grid, axis=0, kind='linear', fill_value="extrapolate"
#     )
    
#     r_tail = radii_interpolator(1.0)

#     # -------------------------------------------------------------------------
#     # 2. FRAME GENERATION (DOWNSAMPLED SLICES)
#     # -------------------------------------------------------------------------
#     all_v1 = []
#     all_v2 = []
#     n_total_full = len(smoothed_centerline)

#     # Zone 1: Pure native frames on downsampled indices
#     for idx in zone1_indices:
#         dir_vec = smoothed_centerline[min(idx + 1, n_total_full - 1)] - smoothed_centerline[max(idx - 1, 0)]
#         v1, v2 = get_local_frame(dir_vec)
#         all_v1.append(v1)
#         all_v2.append(v2)

#     # Zone 2: Parallel Transport extension on downsampled indices
#     for idx in zone2_indices:
#         dir_vec = smoothed_centerline[min(idx + 1, n_total_full - 1)] - smoothed_centerline[max(idx - 1, 0)]
#         norm_dir = numpy.linalg.norm(dir_vec)
#         t_curr = dir_vec / norm_dir if norm_dir > 0 else numpy.array([1.0, 0.0, 0.0])

#         v1_prev = all_v1[-1]
#         v1_next = v1_prev - numpy.dot(v1_prev, t_curr) * t_curr
#         norm_v1 = numpy.linalg.norm(v1_next)
#         if norm_v1 > 1e-8:
#             v1_next /= norm_v1
#         else:
#             v1_next = v1_prev

#         v2_next = numpy.cross(t_curr, v1_next)
#         v2_next /= numpy.linalg.norm(v2_next)

#         all_v1.append(v1_next)
#         all_v2.append(v2_next)

#     # -------------------------------------------------------------------------
#     # 3. ALIGN EYE RING PHASE
#     # -------------------------------------------------------------------------
#     p_touchdown = smoothed_centerline[-1]
#     v1_end, v2_end = all_v1[-1], all_v2[-1]

#     raw_eye_offsets = eye_wall_ring_vox - p_touchdown[None, :]

#     x_proj = numpy.dot(raw_eye_offsets, v1_end)
#     y_proj = numpy.dot(raw_eye_offsets, v2_end)
    
#     eye_angles = numpy.arctan2(y_proj, x_proj)
#     eye_radii = numpy.sqrt(x_proj**2 + y_proj**2)

#     sort_idx = numpy.argsort(eye_angles)
#     sorted_angles = eye_angles[sort_idx]
#     sorted_radii = eye_radii[sort_idx]
    
#     sorted_angles_ext = numpy.concatenate([sorted_angles - 2*numpy.pi, sorted_angles, sorted_angles + 2*numpy.pi])
#     sorted_radii_ext = numpy.concatenate([sorted_radii, sorted_radii, sorted_radii])
    
#     r_eye_aligned = scipy.interpolate.interp1d(sorted_angles_ext, sorted_radii_ext, kind='linear')(theta_steps)

#     # -------------------------------------------------------------------------
#     # 4. BUILD GEOMETRY SLICE-BY-SLICE
#     # -------------------------------------------------------------------------
#     zone1_steps_count = max(len(zone1_indices) - 1, 1)
#     zone2_steps_count = max(len(zone2_indices), 1)

#     for k, idx in enumerate(rendered_indices):
#         p_node = smoothed_centerline[idx]
#         slice_theta = theta_steps 
        
#         v1_curr = all_v1[k]
#         v2_curr = all_v2[k]

#         # ZONE 1: CORE MUSCLE BODY
#         if idx <= y_end_idx:
#             t_zone1 = k / zone1_steps_count
#             r_slice = radii_interpolator(t_zone1)
            
#             start_point = smoothed_centerline[y_start_idx]
#             if k < taper_start_len:
#                 t_taper = k / max(taper_start_len - 1, 1)
#                 scale = numpy.sqrt(t_taper * (2.0 - t_taper))  
#                 p_node = (1.0 - t_taper) * start_point + t_taper * p_node
#                 r_slice = r_slice * scale
                
#             r_slice = numpy.maximum(r_slice, 0.05)
            
#             ring = (
#                 p_node[None, :]
#                 + r_slice[:, None] * numpy.cos(slice_theta)[:, None] * v1_curr[None, :]
#                 + r_slice[:, None] * numpy.sin(slice_theta)[:, None] * v2_curr[None, :]
#             )
#             vertices.append(ring)
            
#         # ZONE 2: TRANSITIONAL TUBE
#         elif idx < n_total_full - 1:
#             trans_k = k - len(zone1_indices) + 1
#             t_lin = trans_k / max(zone2_steps_count, 1)
#             t_blend = t_lin ** blend_power
            
#             r_curr = (1.0 - t_blend) * r_tail + t_blend * r_eye_aligned
            
#             ring = (
#                 p_node[None, :]
#                 + r_curr[:, None] * numpy.cos(slice_theta)[:, None] * v1_curr[None, :]
#                 + r_curr[:, None] * numpy.sin(slice_theta)[:, None] * v2_curr[None, :]
#             )
#             vertices.append(ring)
            
#         # ZONE 3: FINAL TOUCHDOWN SEAM
#         else:
#             vertices.append(eye_wall_ring_vox)
        
#     vertices = numpy.vstack(vertices)
    
#     # -------------------------------------------------------------------------
#     # 5. TOPOLOGY ASSEMBLY
#     # -------------------------------------------------------------------------
#     start_tip_idx = len(vertices)
#     vertices = numpy.vstack([vertices, origin_center_vox[None, :]])
    
#     end_tip_idx = len(vertices)
#     vertices = numpy.vstack([vertices, contact_center_vox[None, :]])
    
#     faces = []
    
#     # Proximal Cap
#     for j in range(n_theta):
#         j2 = (j + 1) % n_theta
#         faces.append([start_tip_idx, j2, j])
        
#     # Main Tube Quads
#     for i in range(n_rendered_slices - 1):
#         base0 = i * n_theta
#         base1 = (i + 1) * n_theta
#         for j in range(n_theta):
#             j2 = (j + 1) % n_theta
#             faces.append([base0 + j, base1 + j, base1 + j2])
#             faces.append([base0 + j, base1 + j2, base0 + j2])
            
#     # Distal Cap
#     final_ring_base = (n_rendered_slices - 1) * n_theta
#     for j in range(n_theta):
#         j2 = (j + 1) % n_theta
#         faces.append([final_ring_base + j, end_tip_idx, final_ring_base + j2])
        
    
    

#     # -------------------------------------------------------------------------
#     # 5. TOPOLOGY ASSEMBLY & REPAIR FOR TETGEN
#     # -------------------------------------------------------------------------
#     mesh = trimesh.Trimesh(
#         vertices=vertices * pixel_spacing_val,
#         faces=numpy.asarray(faces),
#         process=True
#     )

#     import pymeshfix

#     # -------------------------------------------------------------------------
#     # 5. TOPOLOGY ASSEMBLY & FAST SMOOTHING
#     # -------------------------------------------------------------------------
#     mesh = trimesh.Trimesh(
#         vertices=vertices * pixel_spacing_val,
#         faces=numpy.asarray(faces),
#         process=True
#     )

#     # 1. Fill topological open boundary loops
#     trimesh.repair.fill_holes(mesh)

#     v_arr = numpy.ascontiguousarray(mesh.vertices, dtype=numpy.float64)
#     f_arr = numpy.ascontiguousarray(mesh.faces, dtype=numpy.int32)

#     tin = pymeshfix.PyTMesh()
#     tin.load_array(v_arr, f_arr)
#     tin.clean()
#     tin.fill_small_boundaries()

#     mesh = trimesh.Trimesh(
#         vertices=tin.return_points(),
#         faces=tin.return_faces(),
#         process=True
#     )

#     return mesh
    

In [25]:
def generate_blended_mesh(
    full_centerline, 
    short_centerline, 
    short_radii_grid, 
    eye_wall_ring, 
    contact_center_mm, 
    pixel_spacing_val, 
    origin_center_mm=None,
    spine_smoothing=5.0,
    blend_power=2.0,
    stride=1,
    skip_penultimate_slices=0  # <--- Drops 1 (or 2) slices right before the eye wall ring
):
    """
    Generates smooth muscle mesh using continuous arc-length parameter interpolation.
    Explicitly drops `skip_penultimate_slices` right before the final contact surface
    to avoid ring collision, while keeping `eye_wall_ring` as the exact final boundary.
    """
    n_real_slices, n_theta = short_radii_grid.shape    
    vertices = []
    
    theta_steps = numpy.linspace(-numpy.pi, numpy.pi, n_theta, endpoint=False)
    
    eye_wall_ring_vox = eye_wall_ring / pixel_spacing_val
    contact_center_vox = contact_center_mm / pixel_spacing_val
    
    # -------------------------------------------------------------------------
    # 1. ANCHOR MAPPING & CONTINUOUS PARAMETERIZATION
    # -------------------------------------------------------------------------
    full_tree = scipy.spatial.KDTree(full_centerline)
    
    _, y_start_idx = full_tree.query(short_centerline[0])
    _, y_end_idx = full_tree.query(short_centerline[-1])
    
    smoothed_centerline = smooth_centerline_segment(full_centerline, y_end_idx, smoothing_factor=spine_smoothing)
    
    # --- DOWNSAMPLE SLICES & SKIP PENULTIMATE INTERSECTING SLICES ---
    zone1_indices = list(range(y_start_idx, y_end_idx, stride))
    if y_end_idx not in zone1_indices:
        zone1_indices.append(y_end_idx)

    # Build Zone 2 indices up to (end - 1 - skip_penultimate_slices)
    last_centerline_idx = len(smoothed_centerline) - 1
    cutoff_idx = max(last_centerline_idx - 1 - skip_penultimate_slices, y_end_idx + 1)
    
    zone2_indices = list(range(y_end_idx + stride, cutoff_idx + 1, stride))
    if cutoff_idx not in zone2_indices and cutoff_idx > y_end_idx:
        zone2_indices.append(cutoff_idx)

    # rendered_indices contains body slices + the final centerline index (for the eye wall)
    rendered_indices = zone1_indices + zone2_indices + [last_centerline_idx]
    n_rendered_slices = len(rendered_indices)
    
    if origin_center_mm is None:
        origin_center_vox = smoothed_centerline[y_start_idx]
    else:
        origin_center_vox = origin_center_mm / pixel_spacing_val
    
    taper_start_len = max(int(12 / stride), 1)

    # Build continuous interpolation for Zone 1 radii along arc-length
    t_short = numpy.linspace(0, 1, n_real_slices)
    radii_interpolator = scipy.interpolate.interp1d(
        t_short, short_radii_grid, axis=0, kind='linear', fill_value="extrapolate"
    )
    
    r_tail = radii_interpolator(1.0)

    # -------------------------------------------------------------------------
    # 2. FRAME GENERATION
    # -------------------------------------------------------------------------
    all_v1 = []
    all_v2 = []
    n_total_full = len(smoothed_centerline)

    # Zone 1: Pure native frames on downsampled indices
    for idx in zone1_indices:
        dir_vec = smoothed_centerline[min(idx + 1, n_total_full - 1)] - smoothed_centerline[max(idx - 1, 0)]
        v1, v2 = get_local_frame(dir_vec)
        all_v1.append(v1)
        all_v2.append(v2)

    # Zone 2: Parallel Transport extension
    for idx in zone2_indices:
        dir_vec = smoothed_centerline[min(idx + 1, n_total_full - 1)] - smoothed_centerline[max(idx - 1, 0)]
        norm_dir = numpy.linalg.norm(dir_vec)
        t_curr = dir_vec / norm_dir if norm_dir > 0 else numpy.array([1.0, 0.0, 0.0])

        v1_prev = all_v1[-1]
        v1_next = v1_prev - numpy.dot(v1_prev, t_curr) * t_curr
        norm_v1 = numpy.linalg.norm(v1_next)
        if norm_v1 > 1e-8:
            v1_next /= norm_v1
        else:
            v1_next = v1_prev

        v2_next = numpy.cross(t_curr, v1_next)
        v2_next /= numpy.linalg.norm(v2_next)

        all_v1.append(v1_next)
        all_v2.append(v2_next)

    # Final frame at touchdown for eye wall phase alignment
    dir_vec_end = smoothed_centerline[-1] - smoothed_centerline[max(len(smoothed_centerline) - 2, 0)]
    norm_end = numpy.linalg.norm(dir_vec_end)
    t_end = dir_vec_end / norm_end if norm_end > 0 else numpy.array([1.0, 0.0, 0.0])
    v1_end = all_v1[-1] - numpy.dot(all_v1[-1], t_end) * t_end
    if numpy.linalg.norm(v1_end) > 1e-8:
        v1_end /= numpy.linalg.norm(v1_end)
    else:
        v1_end = all_v1[-1]
    v2_end = numpy.cross(t_end, v1_end)
    v2_end /= numpy.linalg.norm(v2_end)

    all_v1.append(v1_end)
    all_v2.append(v2_end)

    # -------------------------------------------------------------------------
    # 3. ALIGN EYE RING PHASE
    # -------------------------------------------------------------------------
    p_touchdown = smoothed_centerline[-1]

    raw_eye_offsets = eye_wall_ring_vox - p_touchdown[None, :]

    x_proj = numpy.dot(raw_eye_offsets, v1_end)
    y_proj = numpy.dot(raw_eye_offsets, v2_end)
    
    eye_angles = numpy.arctan2(y_proj, x_proj)
    eye_radii = numpy.sqrt(x_proj**2 + y_proj**2)

    sort_idx = numpy.argsort(eye_angles)
    sorted_angles = eye_angles[sort_idx]
    sorted_radii = eye_radii[sort_idx]
    
    sorted_angles_ext = numpy.concatenate([sorted_angles - 2*numpy.pi, sorted_angles, sorted_angles + 2*numpy.pi])
    sorted_radii_ext = numpy.concatenate([sorted_radii, sorted_radii, sorted_radii])
    
    r_eye_aligned = scipy.interpolate.interp1d(sorted_angles_ext, sorted_radii_ext, kind='linear')(theta_steps)

    # -------------------------------------------------------------------------
    # 4. BUILD GEOMETRY SLICE-BY-SLICE
    # -------------------------------------------------------------------------
    zone1_steps_count = max(len(zone1_indices) - 1, 1)
    # Step count across Zone 2 plus the direct bridge to the final eye wall ring
    zone2_total_steps = max(len(zone2_indices) + 1, 1)

    for k, idx in enumerate(rendered_indices):
        p_node = smoothed_centerline[idx]
        slice_theta = theta_steps 
        
        v1_curr = all_v1[k]
        v2_curr = all_v2[k]

        # ZONE 1: CORE MUSCLE BODY
        if idx <= y_end_idx:
            t_zone1 = k / zone1_steps_count
            r_slice = radii_interpolator(t_zone1)
            
            start_point = smoothed_centerline[y_start_idx]
            if k < taper_start_len:
                t_taper = k / max(taper_start_len - 1, 1)
                scale = numpy.sqrt(t_taper * (2.0 - t_taper))  
                p_node = (1.0 - t_taper) * start_point + t_taper * p_node
                r_slice = r_slice * scale
                
            r_slice = numpy.maximum(r_slice, 0.05)
            
            ring = (
                p_node[None, :]
                + r_slice[:, None] * numpy.cos(slice_theta)[:, None] * v1_curr[None, :]
                + r_slice[:, None] * numpy.sin(slice_theta)[:, None] * v2_curr[None, :]
            )
            vertices.append(ring)
            
        # ZONE 2: TRANSITIONAL TUBE (Up to slice N - 1 - skip_penultimate_slices)
        elif k < len(rendered_indices) - 1:
            trans_k = k - len(zone1_indices) + 1
            t_lin = trans_k / zone2_total_steps
            t_blend = t_lin ** blend_power
            
            r_curr = (1.0 - t_blend) * r_tail + t_blend * r_eye_aligned
            
            ring = (
                p_node[None, :]
                + r_curr[:, None] * numpy.cos(slice_theta)[:, None] * v1_curr[None, :]
                + r_curr[:, None] * numpy.sin(slice_theta)[:, None] * v2_curr[None, :]
            )
            vertices.append(ring)
            
        # ZONE 3: FINAL TOUCHDOWN SEAM (Attached directly to eye_wall_ring_vox)
        else:
            vertices.append(eye_wall_ring_vox)
        
    vertices = numpy.vstack(vertices)
    
    # -------------------------------------------------------------------------
    # 5. TOPOLOGY ASSEMBLY
    # -------------------------------------------------------------------------
    start_tip_idx = len(vertices)
    vertices = numpy.vstack([vertices, origin_center_vox[None, :]])
    
    end_tip_idx = len(vertices)
    vertices = numpy.vstack([vertices, contact_center_vox[None, :]])
    
    faces = []
    
    # Proximal Cap
    for j in range(n_theta):
        j2 = (j + 1) % n_theta
        faces.append([start_tip_idx, j2, j])
        
    # Main Tube Quads (Now connects cutoff slice DIRECTLY to eye_wall_ring_vox)
    for i in range(n_rendered_slices - 1):
        base0 = i * n_theta
        base1 = (i + 1) * n_theta
        for j in range(n_theta):
            j2 = (j + 1) % n_theta
            faces.append([base0 + j, base1 + j, base1 + j2])
            faces.append([base0 + j, base1 + j2, base0 + j2])
            
    # Distal Cap
    final_ring_base = (n_rendered_slices - 1) * n_theta
    for j in range(n_theta):
        j2 = (j + 1) % n_theta
        faces.append([final_ring_base + j, end_tip_idx, final_ring_base + j2])

    # -------------------------------------------------------------------------
    # 6. TOPOLOGY CLEANUP & REPAIR
    # -------------------------------------------------------------------------
    mesh = trimesh.Trimesh(
        vertices=vertices * pixel_spacing_val,
        faces=numpy.asarray(faces),
        process=True
    )

    trimesh.repair.fill_holes(mesh)
    trimesh.smoothing.filter_taubin(mesh, lamb=0.5, nu=-0.53, iterations=6)

    import pymeshfix
    v_arr = numpy.ascontiguousarray(mesh.vertices, dtype=numpy.float64)
    f_arr = numpy.ascontiguousarray(mesh.faces, dtype=numpy.int32)

    tin = pymeshfix.PyTMesh()
    tin.load_array(v_arr, f_arr)
    tin.clean()
    tin.fill_small_boundaries()

    mesh = trimesh.Trimesh(
        vertices=tin.return_points(),
        faces=tin.return_faces(),
        process=True
    )

    return mesh

## Meshing loop - 2

In [26]:
blend_window_slices = 10  # Smooth transition run-up zone inside the muscle body

# for muscle_id in range(len(muslce_ids)):

for muscle_id in range(1,2):

    print(f"Processing Unpolluted Asset Pipeline for Muscle: {muslce_ids[muscle_id]}")

    eye_intersection_idx = eye_intersection_IDs[muscle_id]
    short_centerline = centerlines[muscle_id] # Length N
    
    # Trim extended path exactly to the eye surface touchdown (Length M)
    centerline_trimmed = extended_centerlines_1[muscle_id][:eye_intersection_idx + 1]
    target_eye_wall_ring = eye_surfaces[muscle_id]
    contact_center = contact_centers[muscle_id]

    # Step 1: Sweep raw cross-sections ONLY along the real segmented short path (N)
    radii_raw = extract_radial_profiles_from_mask(
        centerline=short_centerline,
        mask=muscles[muscle_id],
        pixel_spacing=pixel_spacing,
        n_theta=n_theta_points
    )
    

    radii_pure_smoothed = smooth_and_extrapolate_radii(
        radii_grid=radii_raw,
        n_output_extended=len(short_centerline),
        smoothing=15.0  # Adjust 's' to make the linear trends stiffer or looser
    )


    # Step 3: Generate the mesh by bridging N data directly to the M target mesh ring
    # Compile the closed mesh
    mesh = generate_blended_mesh(
        full_centerline=centerline_trimmed, 
        short_centerline= short_centerline,
        short_radii_grid=radii_pure_smoothed,  
        eye_wall_ring=target_eye_wall_ring,    
        contact_center_mm=contact_center,      # Pass the absolute millimeter center point here
        pixel_spacing_val=pixel_spacing[0]
    )


    
    # Export the clean, artifact-free muscle asset
    output_filename = f"{export_name}_BLENDED_id_{muslce_ids[muscle_id]}.stl"
    mesh.export(output_filename)
    print(f"Successfully generated clean mesh: {output_filename}\n")


    # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 
    
    out_folder = "re-meshed_alt-1_h_"+str(h).replace(".","-")+"/"

    if not os.path.isdir(base + folder + out_folder):
        os.mkdir(base + folder + out_folder)

        
    stl_file = output_filename
    output_filename = base + folder + out_folder + "remeshed_h_"+str(h).replace(".","-") + "_id_"+ str(muslce_ids[muscle_id]) 

    
    tetra_mesh_from_stl(stl_file, output_filename, h)
    set_the_offset(output_filename, offset)



Processing Unpolluted Asset Pipeline for Muscle: 5

Loading ..98%
INFO- ********* ITERATION 0 *********
INFO- Removing degeneracies...
INFO- Removing self-intersections...

99 % done   
INFO- 192 intersecting triangles have been selected.

0 % done   
INFO- 26 intersecting triangles have been selected.

0 % done   
INFO- No intersections detected.

0% done 
Successfully generated clean mesh: /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Radial-profiles-from-mask/segmentations_pdt1_v160725._BLENDED_id_5.stl

Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Radial-profiles-from-mask/segmentations_pdt1_v160725._BLENDED_id_5.stl'...
Info    : Mesh is in binary format
Info    : 12678 facets in solid 0 
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Radial-profiles-from-mask/segmentations_pdt1_v160725._BLENDED_id_5.stl'
Info    : Classifying surfaces (angle: 89)...
Info    : Spli

Warning: STL can only write triangle cells. Discarding tetra, line, vertex.

In [30]:
output_filename = f"{export_name}_BLENDED_id_{muslce_ids[eyeball_id]}.stl"
mesh.export(output_filename)
print(f"Successfully generated clean mesh: {output_filename}\n")    

stl_file = export_name + "_surf_id_"+ str(eyeball_id) + ".stl"

output_filename = base + folder + out_folder + "remeshed_h_"+str(h).replace(".","-") + "_id_"+ str(eyeball_id) + "_shell"

tetra_shell_from_two_surfaces(stl_file.replace(".stl", "_outer.stl"), stl_file.replace(".stl", "_inner.stl"), output_filename, h)
set_the_offset(output_filename, offset)


stl_file = "/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Radial-profiles-from-mask/segmentations_pdt1_v160725._surf_id_2.stl"
id = 2
output_filename = base + folder + out_folder + "remeshed_h_"+str(h).replace(".","-") + "_id_"+ str(id) 

tetra_mesh_from_stl(stl_file, output_filename, h)
set_the_offset(output_filename, offset)


Successfully generated clean mesh: /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Radial-profiles-from-mask/segmentations_pdt1_v160725._BLENDED_id_7.stl

Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Radial-profiles-from-mask/segmentations_pdt1_v160725._surf_id_3_outer.stl'...
Info    : 60200 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Radial-profiles-from-mask/segmentations_pdt1_v160725._surf_id_3_outer.stl'
Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Radial-profiles-from-mask/segmentations_pdt1_v160725._surf_id_3_inner.stl'...
Info    : 60200 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Radial-profiles-from-mask/segmentations_pdt1_v160725._surf_id_3_inner.s

Warning: STL can only write triangle cells. Discarding tetra, line, vertex.

Info    : Reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Radial-profiles-from-mask/segmentations_pdt1_v160725._surf_id_2.stl'...
Info    : 3564 facets in solid 0 Visualization Toolkit generated SLA File
Info    : Done reading '/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/cropped T1/Radial-profiles-from-mask/segmentations_pdt1_v160725._surf_id_2.stl'
Info    : Classifying surfaces (angle: 89)...
Info    : Splitting triangulations to make them parametrizable:
Info    :  - Level 0 partition with 3564 triangles split in 2 parts because poincare characteristic 2 is not 0
Info    : Found 2 model surfaces
Info    : Found 1 model curves
Info    : Done classifying surfaces (Wall 0.0317007s, CPU 0.031527s)
Info    : Creating geometry of discrete curves...
Info    : Done creating geometry of discrete curves (Wall 5.95907e-06s, CPU 5e-06s)
Info    : Creating geometry of discrete surfaces...
Info    : Done creating geometry of discrete surfaces (Wall 

Warning: STL can only write triangle cells. Discarding tetra, line, vertex.

# Visualisation & Testing

In [28]:

# def debug_filtered_radial_slice(centerline, mask, pixel_spacing, slice_idx, n_theta=64, filter_size=5):
#     """
#     Visualizes a single 2D slice with an interactive option to test cyclic 
#     intraslice filtering (median + mean) to eliminate boundary spikes.
#     """
#     voxel_coords = numpy.argwhere(mask)
#     if len(voxel_coords) == 0:
#         print("Mask is completely empty.")
#         return
        
#     p = centerline[slice_idx]
    
#     # 1. Local tracking frame setup
#     direction = centerline[min(slice_idx+1, len(centerline)-1)] - centerline[max(slice_idx-1, 0)]
#     norm_dir = numpy.linalg.norm(direction)
#     if norm_dir == 0:
#         print(f"Slice {slice_idx} has a zero tangent vector.")
#         return
#     direction /= norm_dir
#     v1, v2 = get_local_frame(direction)
    
#     # 2. Extract neighbor plane voxels (+/- 1.0 voxel slab)
#     distances = numpy.linalg.norm(voxel_coords - p, axis=1)
#     close_mask = distances < 45
#     local_voxels = voxel_coords[close_mask] - p
    
#     dist_plane = numpy.dot(local_voxels, direction)
#     slab = numpy.abs(dist_plane) < 1.0
#     local_voxels = local_voxels[slab]
    
#     if len(local_voxels) < 5:
#         print(f"Slice {slice_idx} does not intersect enough mask voxels.")
#         return
        
#     x = local_voxels @ v1
#     y = local_voxels @ v2
    
#     vox_theta = numpy.arctan2(y, x)
#     vox_r = numpy.sqrt(x**2 + y**2)
    
#     # 3. Extract Raw Radar Sweep Radii
#     theta_steps = numpy.linspace(-numpy.pi, numpy.pi, n_theta, endpoint=False)
#     raw_radii = numpy.zeros(n_theta)
    
#     for t_idx, target_theta in enumerate(theta_steps):
#         angle_diff = numpy.abs(numpy.arctan2(numpy.sin(vox_theta - target_theta), numpy.cos(vox_theta - target_theta)))
#         angular_slice_mask = angle_diff < (numpy.pi / 30)
        
#         if numpy.any(angular_slice_mask):
#             raw_radii[t_idx] = numpy.max(vox_r[angular_slice_mask])
#         else:
#             raw_radii[t_idx] = numpy.nan
            
#     # Clean up isolated missing rays using linear interpolation across the loop
#     nans = numpy.isnan(raw_radii)
#     if numpy.any(nans):
#         # Handle wrapping boundary interpolation safely
#         xp = numpy.where(~nans)[0]
#         fp = raw_radii[~nans]
#         raw_radii[nans] = numpy.interp(numpy.where(nans)[0], xp, fp, period=n_theta)

#     # -------------------------------------------------------------------------
#     # STEP 4: APPLY THE CYCLIC FILTER
#     # -------------------------------------------------------------------------
#     smooth_radii = median_filter(raw_radii, size=filter_size, mode='wrap')
#     smooth_radii = uniform_filter1d(smooth_radii, size=filter_size, mode='wrap')

#     # Convert both raw and smoothed profiles back to 2D coordinates for plotting
#     raw_x = raw_radii * numpy.cos(theta_steps)
#     raw_y = raw_radii * numpy.sin(theta_steps)
#     raw_x = numpy.append(raw_x, raw_x[0]) # close loop
#     raw_y = numpy.append(raw_y, raw_y[0])

#     smooth_x = smooth_radii * numpy.cos(theta_steps)
#     smooth_y = smooth_radii * numpy.sin(theta_steps)
#     smooth_x = numpy.append(smooth_x, smooth_x[0]) # close loop
#     smooth_y = numpy.append(smooth_y, smooth_y[0])

#     # -------------------------------------------------------------------------
#     # 5. Plotting Comparison
#     # -------------------------------------------------------------------------
#     plt.figure(figsize=(9, 9))
    
#     # Draw raw voxel matrix background
#     plt.scatter(x, y, color='royalblue', s=45, marker='s', alpha=0.3, label='Raw Mask Pixels')
#     plt.plot(0, 0, 'go', markersize=10, zorder=5, label='Centerline Center (0,0)')
    
#     # Draw the spiked raw footprint
#     plt.plot(raw_x, raw_y, color='orange', linewidth=1.2, linestyle='--', label='Raw Spiked Footprint')
    
#     # Draw the beautifully continuous filtered boundary path
#     plt.plot(smooth_x, smooth_y, color='crimson', linewidth=2.5, label=f'Filtered Contour (size={filter_size})')
#     plt.scatter(smooth_x[:-1], smooth_y[:-1], color='crimson', s=20, zorder=4) # Final vertex dots
    
#     plt.axhline(0, color='black', linewidth=0.5, linestyle='--')
#     plt.axvline(0, color='black', linewidth=0.5, linestyle='--')
#     plt.axis('equal')
#     plt.title(f"Intraslice Filtering Check — Muscle Slice Index: {slice_idx}")
#     plt.xlabel("Local X (v1)")
#     plt.ylabel("Local Y (v2)")
#     plt.legend(loc='upper right')
#     plt.grid(True, linestyle=':', alpha=0.3)
#     plt.show()


# filter_size=5

# target_id = 3
# total_slices = len(centerlines[target_id])

# # Run it on the exact slice that was warping your 3D mesh
# debug_filtered_radial_slice(
#     centerlines[target_id], 
#     muscles[target_id], 
#     pixel_spacing, 
#     slice_idx=total_slices // 2, 
#     n_theta=64, 
#     filter_size=filter_size
# )




In [29]:
# def debug_global_mask_median_vs_angular(centerline, raw_mask, pre_filtered_mask, pixel_spacing, slice_idx, n_theta=64, filter_size=5):
#     """
#     Compares:
#     Left: Raw mask sliced -> angular cyclic 'spider' filter applied to the radii.
#     Right: Globally pre-filtered mask sliced -> raw radar sweep (no further filtering).
#     """
#     p = centerline[slice_idx]
    
#     # 1. Setup Local Tracking Frame
#     direction = centerline[min(slice_idx+1, len(centerline)-1)] - centerline[max(slice_idx-1, 0)]
#     norm_dir = numpy.linalg.norm(direction)
#     if norm_dir == 0:
#         print(f"Slice {slice_idx} has a zero tangent vector.")
#         return
#     direction /= norm_dir
#     v1, v2 = get_local_frame(direction)
    
#     # 2. Helper to extract local slice coordinates from a given mask
#     def get_slice_coordinates(mask_volume):
#         voxel_coords = numpy.argwhere(mask_volume)
#         if len(voxel_coords) == 0:
#             return numpy.array([]), numpy.array([])
        
#         distances = numpy.linalg.norm(voxel_coords - p, axis=1)
#         close_mask = distances < 45
#         local_voxels = voxel_coords[close_mask] - p
        
#         dist_plane = numpy.dot(local_voxels, direction)
#         slab = numpy.abs(dist_plane) < 1.0
#         local_voxels = local_voxels[slab]
        
#         if len(local_voxels) < 5:
#             return numpy.array([]), numpy.array([])
            
#         px = local_voxels @ v1
#         py = local_voxels @ v2
#         return px, py

#     # 3. Helper to extract the radar footprint
#     def extract_footprint(px, py, theta_steps):
#         if len(px) == 0:
#             return numpy.zeros(len(theta_steps))
#         v_theta = numpy.arctan2(py, px)
#         v_r = numpy.sqrt(px**2 + py**2)
#         radii = numpy.zeros(len(theta_steps))
        
#         for t_idx, target_theta in enumerate(theta_steps):
#             angle_diff = numpy.abs(numpy.arctan2(numpy.sin(v_theta - target_theta), numpy.cos(v_theta - target_theta)))
#             angular_slice_mask = angle_diff < (numpy.pi / 30)
#             if numpy.any(angular_slice_mask):
#                 radii[t_idx] = numpy.max(v_r[angular_slice_mask])
#             else:
#                 radii[t_idx] = numpy.nan
                
#         nans = numpy.isnan(radii)
#         if numpy.any(nans) and not numpy.all(nans):
#             xp = numpy.where(~nans)[0]
#             fp = radii[~nans]
#             radii[nans] = numpy.interp(numpy.where(nans)[0], xp, fp, period=len(theta_steps))
#         elif numpy.all(nans):
#             radii[:] = 1.0
#         return radii

#     theta_steps = numpy.linspace(-numpy.pi, numpy.pi, n_theta, endpoint=False)
    
#     # Extract coordinates for both masks on this plane
#     raw_x, raw_y = get_slice_coordinates(raw_mask)
#     filtered_x, filtered_y = get_slice_coordinates(pre_filtered_mask)
    
#     if len(raw_x) == 0 or len(filtered_x) == 0:
#         print(f"Slice {slice_idx} lacks enough overlapping voxels in one of the masks.")
#         return

#     # --- Left Plot Line: Your Angular Flow ---
#     raw_radii = extract_footprint(raw_x, raw_y, theta_steps)
#     smooth_radii = median_filter(raw_radii, size=filter_size, mode='wrap')
#     smooth_radii = uniform_filter1d(smooth_radii, size=filter_size, mode='wrap')

#     # --- Right Plot Line: Raw Sweep of the Pre-Filtered Mask ---
#     global_filtered_radii = extract_footprint(filtered_x, filtered_y, theta_steps)

#     # Prepare closed loops for rendering
#     def close_loop(r):
#         cx = r * numpy.cos(theta_steps)
#         cy = r * numpy.sin(theta_steps)
#         return numpy.append(cx, cx[0]), numpy.append(cy, cy[0])

#     raw_x_loop, raw_y_loop = close_loop(raw_radii)
#     smooth_x_loop, smooth_y_loop = close_loop(smooth_radii)
#     global_f_x_loop, global_f_y_loop = close_loop(global_filtered_radii)

#     # -------------------------------------------------------------------------
#     # PLOT RENDERING
#     # -------------------------------------------------------------------------
#     fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8), sharex=True, sharey=True)
    
#     # --- LEFT PANEL: YOUR ANGULAR PIPELINE ---
#     ax1.scatter(raw_x, raw_y, color='royalblue', s=45, marker='s', alpha=0.3, label='Raw Mask Pixels')
#     ax1.plot(0, 0, 'go', markersize=10, zorder=5)
#     ax1.plot(raw_x_loop, raw_y_loop, color='orange', linewidth=1.2, linestyle='--', label='Raw Spiked Sweep')
#     ax1.plot(smooth_x_loop, smooth_y_loop, color='crimson', linewidth=2.5, label='Angular "Wrap" Filter')
#     ax1.scatter(smooth_x_loop[:-1], smooth_y_loop[:-1], color='crimson', s=15, zorder=4)
#     ax1.set_title("Your Method: Post-Extraction Angular Filtering\n(Sub-voxel precision, smooth continuity)", fontsize=11)
#     ax1.grid(True, linestyle=':', alpha=0.4)
#     ax1.legend(loc='upper right')
#     ax1.set_xlabel("Local X (v1)")
#     ax1.set_ylabel("Local Y (v2)")
#     # ax1.axis('equal')

#     # --- RIGHT PANEL: PRE-FILTERED MASK ---
#     ax2.scatter(raw_x, raw_y, color='royalblue', s=45, marker='s', alpha=0.3, label='Raw Mask Pixels')
#     ax2.scatter(filtered_x, filtered_y, color='forestgreen', s=45, marker='s', alpha=0.4, label='Pre-Filtered Mask Pixels')
#     ax2.plot(0, 0, 'go', markersize=10, zorder=5)
#     ax2.plot(global_f_x_loop, global_f_y_loop, color='darkgreen', linewidth=2.5, label='Raw Sweep Footprint')
#     ax2.scatter(global_f_x_loop[:-1], global_f_y_loop[:-1], color='darkgreen', s=15, zorder=4)
#     ax2.set_title(f"Alternative: Global Mask Median Filter (size={filter_size}) applied first\n(Traces original voxel blockiness directly)", fontsize=11)
#     ax2.grid(True, linestyle=':', alpha=0.4)
#     ax2.legend(loc='upper right')
#     ax2.set_xlabel("Local X (v1)")

#     plt.suptitle(f"Anatomical Preservation Check — Muscle Slice Index: {slice_idx}", fontsize=14, weight='bold')
#     plt.tight_layout()
#     plt.show()



# # DEFINE CONFIGURATION CONSTANTS
# # -------------------------------------------------------------------------
# filter_size = 5     # Size of the structural footprint kernel
# n_theta_points = 64 # Angular ray density
# target_id = 3       # The index of the specific extraocular muscle asset you are testing

# # Calculate explicit tracking index markers
# total_slices = len(centerlines[target_id])
# quarter_slice = total_slices // 4
# mid_slice     = total_slices // 2
# three_quarter = (3 * total_slices) // 4

# # -------------------------------------------------------------------------
# # RUN COMPARATIVE VISUALIZATIONS
# # -------------------------------------------------------------------------
# target_id = 3
# filter_size = 5

# # 1. Generate the globally filtered volume mask first
# globally_smoothed_mask = median_filter(muscles[target_id], size=9)

# total_slices = len(centerlines[target_id])

# # 2. Run the comparison on the target slice
# debug_global_mask_median_vs_angular(
#     centerline=centerlines[target_id],
#     raw_mask=muscles[target_id],
#     pre_filtered_mask=globally_smoothed_mask,
#     pixel_spacing=pixel_spacing,
#     slice_idx=total_slices // 2,
#     n_theta=64,
#     filter_size=filter_size
# )